# Agent 2 — PMT Topical QP/MS Question-Level Parsing

## Notebook 02 FINAL: Question parsing, mark-scheme parsing, alignment, and PostgreSQL storage

This version was updated after inspecting the complete PMT topical PDF cache.

It supports the different AQA/PMT layouts found across all 35 QP/MS pairs:

- QP question numbers split into separate PDF text fragments;
- repeated question headers on continuation pages;
- QP context records followed by scored subparts;
- MS tables with and without visible column headings;
- MS entries where total marks appear on the first row, another row, or only in AO guidance;
- QP and MS question-number differences inside a topical pack;
- repeated original question numbers from different source papers;
- dynamic sequence alignment instead of unsafe positional zipping;
- low-confidence and unmatched records routed to Human-in-the-Loop review.

### Safety rules

```python
RUN_FULL_BATCH = False
COMMIT_TO_POSTGRES = False
```

The first run is a three-pair pilot.

All parser-generated questions remain:

```text
retrieval_enabled = False
embedding_status = blocked_pending_review
```

Notebook 03 performs human approval/correction before retrieval or Qdrant indexing.


## 1. Install dependencies


In [43]:
%pip install -q pymupdf pandas pydantic python-dotenv "sqlalchemy>=2.0" "psycopg[binary]>=3.1"


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports and configuration


In [44]:
from __future__ import annotations

import hashlib
import json
import os
import re
import time
import uuid
from collections import Counter
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Literal

import fitz
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict, Field
from sqlalchemy import (
    Boolean, CheckConstraint, DateTime, Float, ForeignKey, Index,
    Integer, String, Text, UniqueConstraint, create_engine,
    delete, func, select,
)
from sqlalchemy.dialects.postgresql import JSONB, UUID
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name.lower() in {"notebooks", "notebook"} else cwd
PDF_CACHE = PROJECT_ROOT / "cache" / "pmt_topical_pdfs"
OUTPUT_DIR = PROJECT_ROOT / "OUTPUT"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(PROJECT_ROOT / ".env")
DATABASE_URL = os.getenv("AGENT2_DATABASE_URL", "").strip()
if not DATABASE_URL:
    raise RuntimeError("AGENT2_DATABASE_URL is missing from Agent2/.env")

PARSER_VERSION = "pmt-topical-qp-ms-parser-v2.0.0-final"

# SAFE DEFAULTS
RUN_FULL_BATCH = True
COMMIT_TO_POSTGRES = True
REPROCESS_ALL = False
ALLOW_REPLACE_EXISTING = True
ALLOW_REPLACE_HUMAN_REVIEWED = False

PILOT_PAIR_LIMIT = 3
PILOT_PAIR_KEYS: list[str] = []

EXPECTED_PAIR_COUNT = 35
EXPECTED_DOCUMENT_COUNT = 70

print(f"Project root:      {PROJECT_ROOT}")
print(f"Topical PDF cache: {PDF_CACHE}")
print(f"Output directory:  {OUTPUT_DIR}")
print(f"Parser version:    {PARSER_VERSION}")
print(f"Full batch:        {RUN_FULL_BATCH}")
print(f"Commit to DB:      {COMMIT_TO_POSTGRES}")


Project root:      C:\Users\hp\EDTECH\Agent2
Topical PDF cache: C:\Users\hp\EDTECH\Agent2\cache\pmt_topical_pdfs
Output directory:  C:\Users\hp\EDTECH\Agent2\OUTPUT
Parser version:    pmt-topical-qp-ms-parser-v2.0.0-final
Full batch:        True
Commit to DB:      True


## 3. PostgreSQL models

Notebook 01 already created:

```text
assessment_topical_topics
assessment_topical_documents
```

This notebook creates separate topical parser tables so the archived official-paper dataset is not mixed with the active PMT dataset.


In [45]:
class Base(DeclarativeBase):
    pass


def utc_now() -> datetime:
    return datetime.now(timezone.utc)


# Existing Notebook 01 tables — only the columns used here are mapped.
class AssessmentTopicalTopic(Base):
    __tablename__ = "assessment_topical_topics"

    id: Mapped[uuid.UUID] = mapped_column(UUID(as_uuid=True), primary_key=True)
    topic_key: Mapped[str] = mapped_column(String(180), nullable=False)
    pmt_topic_number: Mapped[int] = mapped_column(Integer, nullable=False)
    pmt_topic_name: Mapped[str] = mapped_column(Text, nullable=False)
    pmt_subtopic_code: Mapped[str] = mapped_column(String(30), nullable=False)
    pmt_subtopic_name: Mapped[str] = mapped_column(Text, nullable=False)
    paper_code: Mapped[str] = mapped_column(String(10), nullable=False)
    programming_language: Mapped[str | None] = mapped_column(String(50))


class AssessmentTopicalDocument(Base):
    __tablename__ = "assessment_topical_documents"

    id: Mapped[uuid.UUID] = mapped_column(UUID(as_uuid=True), primary_key=True)
    topic_id: Mapped[uuid.UUID] = mapped_column(
        UUID(as_uuid=True),
        ForeignKey("assessment_topical_topics.id", ondelete="CASCADE"),
        nullable=False,
    )
    pair_key: Mapped[str] = mapped_column(String(180), nullable=False)
    document_type: Mapped[str] = mapped_column(String(30), nullable=False)
    discovery_title: Mapped[str] = mapped_column(Text, nullable=False)
    source_url: Mapped[str] = mapped_column(Text, nullable=False)
    local_cache_path: Mapped[str | None] = mapped_column(Text)
    file_hash: Mapped[str | None] = mapped_column(String(64))
    page_count: Mapped[int | None] = mapped_column(Integer)
    contains_8520_marker: Mapped[bool] = mapped_column(Boolean, nullable=False)
    contains_8525_marker: Mapped[bool] = mapped_column(Boolean, nullable=False)
    specification_scope: Mapped[str] = mapped_column(String(50), nullable=False)
    download_status: Mapped[str] = mapped_column(String(30), nullable=False)
    parsing_status: Mapped[str] = mapped_column(String(30), nullable=False)
    error_message: Mapped[str | None] = mapped_column(Text)
    updated_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), nullable=False)


REVIEW_STATUS_SQL = (
    "('auto_valid','needs_review','human_approved',"
    "'human_corrected','rejected')"
)


class AssessmentTopicalQuestion(Base):
    __tablename__ = "assessment_topical_questions"
    __table_args__ = (
        UniqueConstraint("question_uid", name="uq_topical_question_uid"),
        UniqueConstraint(
            "question_document_id", "sequence_index",
            name="uq_topical_question_document_sequence",
        ),
        CheckConstraint(
            "record_type IN ('context','scored_item')",
            name="ck_topical_question_record_type",
        ),
        CheckConstraint(
            f"review_status IN {REVIEW_STATUS_SQL}",
            name="ck_topical_question_review_status",
        ),
        Index("ix_topical_question_pair", "pair_key", "sequence_index"),
        Index("ix_topical_question_topic", "topic_id", "retrieval_enabled"),
    )

    id: Mapped[uuid.UUID] = mapped_column(
        UUID(as_uuid=True), primary_key=True, default=uuid.uuid4
    )
    question_uid: Mapped[str] = mapped_column(String(260), nullable=False)
    pair_key: Mapped[str] = mapped_column(String(180), nullable=False)
    topic_id: Mapped[uuid.UUID] = mapped_column(
        UUID(as_uuid=True),
        ForeignKey("assessment_topical_topics.id", ondelete="CASCADE"),
        nullable=False,
    )
    question_document_id: Mapped[uuid.UUID] = mapped_column(
        UUID(as_uuid=True),
        ForeignKey("assessment_topical_documents.id", ondelete="CASCADE"),
        nullable=False,
    )

    sequence_index: Mapped[int] = mapped_column(Integer, nullable=False)
    question_number: Mapped[str] = mapped_column(String(30), nullable=False)
    normalized_question_number: Mapped[str] = mapped_column(String(30), nullable=False)
    occurrence_index: Mapped[int] = mapped_column(Integer, nullable=False)
    main_question_number: Mapped[str] = mapped_column(String(20), nullable=False)
    part_number: Mapped[str | None] = mapped_column(String(20))
    parent_question_number: Mapped[str | None] = mapped_column(String(30))

    record_type: Mapped[str] = mapped_column(String(20), nullable=False)
    question_text: Mapped[str] = mapped_column(Text, nullable=False)
    context_text: Mapped[str] = mapped_column(Text, nullable=False, default="")
    search_text: Mapped[str] = mapped_column(Text, nullable=False)
    raw_extracted_text: Mapped[str] = mapped_column(Text, nullable=False)
    marks: Mapped[int | None] = mapped_column(Integer)

    page_start: Mapped[int] = mapped_column(Integer, nullable=False)
    page_end: Mapped[int] = mapped_column(Integer, nullable=False)
    has_visual: Mapped[bool] = mapped_column(Boolean, nullable=False, default=False)
    visual_page_numbers: Mapped[list[int]] = mapped_column(
        JSONB, nullable=False, default=list
    )
    has_code: Mapped[bool] = mapped_column(Boolean, nullable=False, default=False)

    specification_scope: Mapped[str] = mapped_column(String(60), nullable=False)
    is_legacy: Mapped[bool] = mapped_column(Boolean, nullable=False, default=False)

    parse_warnings: Mapped[list[str]] = mapped_column(
        JSONB, nullable=False, default=list
    )
    review_status: Mapped[str] = mapped_column(
        String(30), nullable=False, default="auto_valid"
    )
    retrieval_enabled: Mapped[bool] = mapped_column(
        Boolean, nullable=False, default=False
    )
    embedding_status: Mapped[str] = mapped_column(
        String(40), nullable=False, default="blocked_pending_review"
    )

    content_hash: Mapped[str] = mapped_column(String(64), nullable=False)
    parse_version: Mapped[str] = mapped_column(String(100), nullable=False)
    is_active: Mapped[bool] = mapped_column(Boolean, nullable=False, default=True)
    created_at: Mapped[datetime] = mapped_column(
        DateTime(timezone=True), nullable=False, default=utc_now
    )
    updated_at: Mapped[datetime] = mapped_column(
        DateTime(timezone=True), nullable=False, default=utc_now, onupdate=utc_now
    )


class AssessmentTopicalMarkSchemeEntry(Base):
    __tablename__ = "assessment_topical_mark_scheme_entries"
    __table_args__ = (
        UniqueConstraint("mark_scheme_uid", name="uq_topical_ms_uid"),
        UniqueConstraint(
            "mark_scheme_document_id", "sequence_index",
            name="uq_topical_ms_document_sequence",
        ),
        CheckConstraint(
            f"review_status IN {REVIEW_STATUS_SQL}",
            name="ck_topical_ms_review_status",
        ),
        Index("ix_topical_ms_pair", "pair_key", "sequence_index"),
    )

    id: Mapped[uuid.UUID] = mapped_column(
        UUID(as_uuid=True), primary_key=True, default=uuid.uuid4
    )
    mark_scheme_uid: Mapped[str] = mapped_column(String(270), nullable=False)
    pair_key: Mapped[str] = mapped_column(String(180), nullable=False)
    topic_id: Mapped[uuid.UUID] = mapped_column(
        UUID(as_uuid=True),
        ForeignKey("assessment_topical_topics.id", ondelete="CASCADE"),
        nullable=False,
    )
    mark_scheme_document_id: Mapped[uuid.UUID] = mapped_column(
        UUID(as_uuid=True),
        ForeignKey("assessment_topical_documents.id", ondelete="CASCADE"),
        nullable=False,
    )

    sequence_index: Mapped[int] = mapped_column(Integer, nullable=False)
    question_number: Mapped[str] = mapped_column(String(30), nullable=False)
    normalized_question_number: Mapped[str] = mapped_column(String(30), nullable=False)
    occurrence_index: Mapped[int] = mapped_column(Integer, nullable=False)
    main_question_number: Mapped[str] = mapped_column(String(20), nullable=False)
    part_number: Mapped[str | None] = mapped_column(String(20))
    maximum_marks: Mapped[int] = mapped_column(Integer, nullable=False)

    marking_guidance: Mapped[str] = mapped_column(Text, nullable=False)
    marking_points: Mapped[list[str]] = mapped_column(JSONB, nullable=False, default=list)
    acceptable_answers: Mapped[list[str]] = mapped_column(JSONB, nullable=False, default=list)
    rejected_answers: Mapped[list[str]] = mapped_column(JSONB, nullable=False, default=list)
    additional_guidance: Mapped[list[str]] = mapped_column(JSONB, nullable=False, default=list)
    assessment_objectives: Mapped[list[str]] = mapped_column(JSONB, nullable=False, default=list)

    page_start: Mapped[int] = mapped_column(Integer, nullable=False)
    page_end: Mapped[int] = mapped_column(Integer, nullable=False)
    raw_extracted_text: Mapped[str] = mapped_column(Text, nullable=False)

    specification_scope: Mapped[str] = mapped_column(String(60), nullable=False)
    is_legacy: Mapped[bool] = mapped_column(Boolean, nullable=False, default=False)
    parse_warnings: Mapped[list[str]] = mapped_column(JSONB, nullable=False, default=list)
    review_status: Mapped[str] = mapped_column(
        String(30), nullable=False, default="auto_valid"
    )

    content_hash: Mapped[str] = mapped_column(String(64), nullable=False)
    parse_version: Mapped[str] = mapped_column(String(100), nullable=False)
    is_active: Mapped[bool] = mapped_column(Boolean, nullable=False, default=True)
    created_at: Mapped[datetime] = mapped_column(
        DateTime(timezone=True), nullable=False, default=utc_now
    )
    updated_at: Mapped[datetime] = mapped_column(
        DateTime(timezone=True), nullable=False, default=utc_now, onupdate=utc_now
    )


class AssessmentTopicalQuestionMarkSchemeLink(Base):
    __tablename__ = "assessment_topical_question_mark_scheme_links"
    __table_args__ = (
        UniqueConstraint("question_id", name="uq_topical_link_question"),
        UniqueConstraint("mark_scheme_entry_id", name="uq_topical_link_ms"),
        CheckConstraint(
            "validation_status IN "
            "('auto_valid','needs_review','human_approved','rejected')",
            name="ck_topical_link_status",
        ),
        Index("ix_topical_link_pair", "pair_key"),
    )

    id: Mapped[uuid.UUID] = mapped_column(
        UUID(as_uuid=True), primary_key=True, default=uuid.uuid4
    )
    pair_key: Mapped[str] = mapped_column(String(180), nullable=False)
    question_id: Mapped[uuid.UUID] = mapped_column(
        UUID(as_uuid=True),
        ForeignKey("assessment_topical_questions.id", ondelete="CASCADE"),
        nullable=False,
    )
    mark_scheme_entry_id: Mapped[uuid.UUID] = mapped_column(
        UUID(as_uuid=True),
        ForeignKey("assessment_topical_mark_scheme_entries.id", ondelete="CASCADE"),
        nullable=False,
    )
    match_method: Mapped[str] = mapped_column(String(50), nullable=False)
    match_confidence: Mapped[float] = mapped_column(Float, nullable=False)
    marks_match: Mapped[bool] = mapped_column(Boolean, nullable=False)
    validation_status: Mapped[str] = mapped_column(String(30), nullable=False)
    validation_warnings: Mapped[list[str]] = mapped_column(
        JSONB, nullable=False, default=list
    )
    created_at: Mapped[datetime] = mapped_column(
        DateTime(timezone=True), nullable=False, default=utc_now
    )


class AssessmentTopicalParsingIssue(Base):
    __tablename__ = "assessment_topical_parsing_issues"
    __table_args__ = (
        CheckConstraint(
            "severity IN ('info','warning','critical')",
            name="ck_topical_issue_severity",
        ),
        Index("ix_topical_issue_pair", "pair_key", "resolved"),
    )

    id: Mapped[uuid.UUID] = mapped_column(
        UUID(as_uuid=True), primary_key=True, default=uuid.uuid4
    )
    pair_key: Mapped[str] = mapped_column(String(180), nullable=False)
    document_id: Mapped[uuid.UUID | None] = mapped_column(
        UUID(as_uuid=True),
        ForeignKey("assessment_topical_documents.id", ondelete="CASCADE"),
    )
    stage: Mapped[str] = mapped_column(String(50), nullable=False)
    issue_type: Mapped[str] = mapped_column(String(100), nullable=False)
    severity: Mapped[str] = mapped_column(String(20), nullable=False)
    description: Mapped[str] = mapped_column(Text, nullable=False)
    details: Mapped[dict[str, Any]] = mapped_column(JSONB, nullable=False, default=dict)
    resolved: Mapped[bool] = mapped_column(Boolean, nullable=False, default=False)
    resolution_note: Mapped[str | None] = mapped_column(Text)
    resolved_by: Mapped[str | None] = mapped_column(String(120))
    resolved_at: Mapped[datetime | None] = mapped_column(DateTime(timezone=True))
    created_at: Mapped[datetime] = mapped_column(
        DateTime(timezone=True), nullable=False, default=utc_now
    )


class AssessmentTopicalParsingRun(Base):
    __tablename__ = "assessment_topical_parsing_runs"

    id: Mapped[uuid.UUID] = mapped_column(
        UUID(as_uuid=True), primary_key=True, default=uuid.uuid4
    )
    parser_version: Mapped[str] = mapped_column(String(100), nullable=False)
    run_mode: Mapped[str] = mapped_column(String(30), nullable=False)
    committed: Mapped[bool] = mapped_column(Boolean, nullable=False, default=False)
    status: Mapped[str] = mapped_column(String(30), nullable=False, default="running")
    started_at: Mapped[datetime] = mapped_column(
        DateTime(timezone=True), nullable=False, default=utc_now
    )
    completed_at: Mapped[datetime | None] = mapped_column(DateTime(timezone=True))
    counts: Mapped[dict[str, Any]] = mapped_column(JSONB, nullable=False, default=dict)
    error_message: Mapped[str | None] = mapped_column(Text)


engine = create_engine(DATABASE_URL, pool_pre_ping=True, future=True)
Base.metadata.create_all(engine)

with engine.connect() as connection:
    connection.exec_driver_sql("SELECT 1")

print("PostgreSQL connection successful.")
print("Created/verified topical parsing tables.")


PostgreSQL connection successful.
Created/verified topical parsing tables.


## 4. Validated parser records


In [46]:
class ParsedQuestion(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True)

    internal_key: str
    question_uid: str
    pair_key: str
    topic_id: uuid.UUID
    question_document_id: uuid.UUID

    sequence_index: int = Field(ge=1)
    question_number: str
    normalized_question_number: str
    occurrence_index: int = Field(ge=1)
    main_question_number: str
    part_number: str | None = None
    parent_question_number: str | None = None

    record_type: Literal["context", "scored_item"]
    question_text: str
    context_text: str = ""
    search_text: str
    raw_extracted_text: str
    marks: int | None = Field(default=None, ge=0)

    page_start: int = Field(ge=1)
    page_end: int = Field(ge=1)
    has_visual: bool = False
    visual_page_numbers: list[int] = Field(default_factory=list)
    has_code: bool = False

    specification_scope: str
    is_legacy: bool = False
    parse_warnings: list[str] = Field(default_factory=list)
    review_status: Literal["auto_valid", "needs_review"] = "auto_valid"

    retrieval_enabled: bool = False
    embedding_status: str = "blocked_pending_review"
    content_hash: str
    parse_version: str = PARSER_VERSION


class ParsedMarkSchemeEntry(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True)

    internal_key: str
    mark_scheme_uid: str
    pair_key: str
    topic_id: uuid.UUID
    mark_scheme_document_id: uuid.UUID

    sequence_index: int = Field(ge=1)
    question_number: str
    normalized_question_number: str
    occurrence_index: int = Field(ge=1)
    main_question_number: str
    part_number: str | None = None
    maximum_marks: int = Field(ge=1)

    marking_guidance: str
    marking_points: list[str] = Field(default_factory=list)
    acceptable_answers: list[str] = Field(default_factory=list)
    rejected_answers: list[str] = Field(default_factory=list)
    additional_guidance: list[str] = Field(default_factory=list)
    assessment_objectives: list[str] = Field(default_factory=list)

    page_start: int = Field(ge=1)
    page_end: int = Field(ge=1)
    raw_extracted_text: str

    specification_scope: str
    is_legacy: bool = False
    parse_warnings: list[str] = Field(default_factory=list)
    review_status: Literal["auto_valid", "needs_review"] = "auto_valid"
    content_hash: str
    parse_version: str = PARSER_VERSION


class ProposedLink(BaseModel):
    question_internal_key: str
    mark_scheme_internal_key: str
    pair_key: str
    match_method: Literal["exact_number_occurrence", "same_main_number_alignment", "sequence_alignment"]
    match_confidence: float = Field(ge=0, le=1)
    marks_match: bool
    validation_status: Literal["auto_valid", "needs_review"]
    validation_warnings: list[str] = Field(default_factory=list)


class PairParseResult(BaseModel):
    pair_key: str
    topic_id: uuid.UUID
    topic_number: int
    topic_name: str
    subtopic_code: str
    subtopic_name: str
    paper_code: str
    programming_language: str | None = None

    question_document_id: uuid.UUID
    mark_scheme_document_id: uuid.UUID
    question_path: str
    mark_scheme_path: str

    questions: list[ParsedQuestion] = Field(default_factory=list)
    mark_scheme_entries: list[ParsedMarkSchemeEntry] = Field(default_factory=list)
    links: list[ProposedLink] = Field(default_factory=list)
    issues: list[dict[str, Any]] = Field(default_factory=list)

    scored_question_count: int = 0
    context_count: int = 0
    mark_scheme_entry_count: int = 0
    exact_link_count: int = 0
    fallback_link_count: int = 0
    unlinked_question_count: int = 0
    unlinked_mark_scheme_count: int = 0
    mark_mismatch_count: int = 0

    strict_passed: bool = False
    needs_human_review: bool = True
    processing_seconds: float = 0.0


@dataclass(slots=True)
class QuestionLine:
    page_number: int
    text: str
    x0: float
    y0: float
    x1: float
    y1: float
    font_sizes: tuple[float, ...] = field(default_factory=tuple)
    question_marker: str | None = None
    marker_remainder: str = ""


@dataclass(slots=True)
class MarkSchemeRow:
    page_number: int
    row_index: int
    y0: float
    y1: float
    text: str
    words: list[tuple]


@dataclass(slots=True)
class MarkSchemeStart:
    page_number: int
    row_index: int
    y: float

    source_main_number: str
    source_part_number: str | None
    source_question_number: str
    source_number_variants: tuple[str, ...]

    row_text: str
    row_words: list[tuple]

    question_word: tuple
    part_word: tuple | None



## 5. Load complete topical QP/MS pairs from PostgreSQL


In [47]:
def load_registered_pairs() -> tuple[list[dict[str, Any]], pd.DataFrame]:
    with Session(engine) as session:
        rows = session.execute(
            select(AssessmentTopicalDocument, AssessmentTopicalTopic)
            .join(
                AssessmentTopicalTopic,
                AssessmentTopicalDocument.topic_id == AssessmentTopicalTopic.id,
            )
            .where(AssessmentTopicalDocument.download_status == "completed")
            .order_by(
                AssessmentTopicalTopic.pmt_topic_number,
                AssessmentTopicalTopic.pmt_subtopic_code,
                AssessmentTopicalDocument.document_type,
            )
        ).all()

    grouped: dict[str, dict[str, Any]] = {}
    issues: list[dict[str, Any]] = []

    for document, topic in rows:
        group = grouped.setdefault(
            document.pair_key,
            {
                "pair_key": document.pair_key,
                "topic_id": topic.id,
                "topic_number": topic.pmt_topic_number,
                "topic_name": topic.pmt_topic_name,
                "subtopic_code": topic.pmt_subtopic_code,
                "subtopic_name": topic.pmt_subtopic_name,
                "paper_code": topic.paper_code,
                "programming_language": topic.programming_language,
                "question_document": None,
                "mark_scheme_document": None,
            },
        )

        if document.document_type == "question_paper":
            if group["question_document"] is not None:
                issues.append({"pair_key": document.pair_key, "issue": "multiple_qp"})
            group["question_document"] = document
        elif document.document_type == "mark_scheme":
            if group["mark_scheme_document"] is not None:
                issues.append({"pair_key": document.pair_key, "issue": "multiple_ms"})
            group["mark_scheme_document"] = document

    complete: list[dict[str, Any]] = []

    for pair_key, group in grouped.items():
        qp = group["question_document"]
        ms = group["mark_scheme_document"]

        if qp is None or ms is None:
            issues.append({"pair_key": pair_key, "issue": "incomplete_pair"})
            continue

        qp_path = Path(qp.local_cache_path or "")
        ms_path = Path(ms.local_cache_path or "")

        missing = [str(p) for p in (qp_path, ms_path) if not p.exists()]
        if missing:
            issues.append(
                {"pair_key": pair_key, "issue": "cache_file_missing", "paths": missing}
            )
            continue

        complete.append(
            {
                "pair_key": pair_key,
                "topic_id": group["topic_id"],
                "topic_number": group["topic_number"],
                "topic_name": group["topic_name"],
                "subtopic_code": group["subtopic_code"],
                "subtopic_name": group["subtopic_name"],
                "paper_code": group["paper_code"],
                "programming_language": group["programming_language"],
                "question_document_id": qp.id,
                "mark_scheme_document_id": ms.id,
                "question_path": qp_path,
                "mark_scheme_path": ms_path,
                "question_parsing_status": qp.parsing_status,
                "mark_scheme_parsing_status": ms.parsing_status,
            }
        )

    return complete, pd.DataFrame(issues)


registered_pairs, source_issues_df = load_registered_pairs()

pairs_df = pd.DataFrame(
    [
        {
            "pair_key": p["pair_key"],
            "topic_number": p["topic_number"],
            "topic": p["topic_name"],
            "subtopic_code": p["subtopic_code"],
            "subtopic": p["subtopic_name"],
            "language": p["programming_language"],
            "qp_path": str(p["question_path"]),
            "ms_path": str(p["mark_scheme_path"]),
        }
        for p in registered_pairs
    ]
)

display(pairs_df)

if not source_issues_df.empty:
    print("Source issues:")
    display(source_issues_df)

print(f"Complete QP/MS pairs: {len(registered_pairs)}")

if len(registered_pairs) != EXPECTED_PAIR_COUNT:
    print(
        f"Warning: expected {EXPECTED_PAIR_COUNT} complete pairs, "
        f"found {len(registered_pairs)}."
    )


,pair_key,topic_number,topic,subtopic_code,subtopic,language,qp_path,ms_path
0,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,1,Fundamentals of Algorithms,1.1,Representing Algorithms,Python,C:\Users\hp\EDTECH\Agent2\cache\pmt_topical_pd...,C:\Users\hp\EDTECH\Agent2\cache\pmt_topical_pd...
1,AQA_GCSE_CS_PMT_T1_1_2_PYTHON,1,Fundamentals of Algorithms,1.2,Efficiency of Algorithms,Python,C:\Users\hp\EDTECH\Agent2\cache\pmt_topical_pd...,C:\Users\hp\EDTECH\Agent2\cache\pmt_topical_pd...
2,AQA_GCSE_CS_PMT_T1_1_3_PYTHON,1,Fundamentals of Algorithms,1.3,Searching Algorithms,Python,C:\Users\hp\EDTECH\Agent2\cache\pmt_topical_pd...,C:\Users\hp\EDTECH\Agent2\cache\pmt_topical_pd...
3,AQA_GCSE_CS_PMT_T1_1_4_PYTHON,1,Fundamentals of Algorithms,1.4,Sorting Algorithms,Python,C:\Users\hp\EDTECH\Agent2\cache\pmt_topical_pd...,C:\Users\hp\EDTECH\Agent2\cache\pmt_topical_pd...
4,AQA_GCSE_CS_PMT_T2_2_01_PYTHON,2,Programming,2.01,Data Types,Python,C:\Users\hp\EDTECH\Agent2\cache\pmt_topical_pd...,C:\Users\hp\EDTECH\Agent2\cache\pmt_topical_pd...
5,AQA_GCSE_CS_PMT_T2_2_02_PYTHON,2,Programming,2.02,Programming Concepts,Python,C:\Users\hp\EDTECH\Agent2\cache\pmt_topical_pd...,C:\Users\hp\EDTECH\Agent2\cache\pmt_topical_pd...
6,AQA_GCSE_CS_PMT_T2_2_03_PYTHON,2,Programming,2.03,Arithmetic Operations,Python,C:\Users\hp\EDTECH\Agent2\cache\pmt_topical_pd...,C:\Users\hp\EDTECH\Agent2\cache\pmt_topical_pd...
7,AQA_GCSE_CS_PMT_T2_2_04_PYTHON,2,Programming,2.04,Relational Operations,Python,C:\Users\hp\EDTECH\Agent2\cache\pmt_topical_pd...,C:\Users\hp\EDTECH\Agent2\cache\pmt_topical_pd...
8,AQA_GCSE_CS_PMT_T2_2_05_PYTHON,2,Programming,2.05,Boolean Operations,Python,C:\Users\hp\EDTECH\Agent2\cache\pmt_topical_pd...,C:\Users\hp\EDTECH\Agent2\cache\pmt_topical_pd...
9,AQA_GCSE_CS_PMT_T2_2_06_PYTHON,2,Programming,2.06,Data Structures,Python,C:\Users\hp\EDTECH\Agent2\cache\pmt_topical_pd...,C:\Users\hp\EDTECH\Agent2\cache\pmt_topical_pd...


Complete QP/MS pairs: 35


## 6. Select the three-pair pilot or all 35 pairs


In [48]:
def select_pilot_pairs(pairs: list[dict[str, Any]]) -> list[dict[str, Any]]:
    by_key = {p["pair_key"]: p for p in pairs}

    if PILOT_PAIR_KEYS:
        missing = [key for key in PILOT_PAIR_KEYS if key not in by_key]
        if missing:
            raise ValueError(f"Unknown PILOT_PAIR_KEYS: {missing}")
        return [by_key[key] for key in PILOT_PAIR_KEYS]

    selected: list[dict[str, Any]] = []

    def add_first(predicate) -> None:
        for pair in pairs:
            if predicate(pair) and pair not in selected:
                selected.append(pair)
                return

    # Programming/Python
    add_first(
        lambda p: p["topic_number"] == 2
        and p["programming_language"] == "Python"
    )

    # Known representative visual/legacy-marker pack
    add_first(
        lambda p: p["topic_number"] == 4
        and "hardware" in p["subtopic_name"].lower()
    )

    # Additional theory category
    add_first(lambda p: p["topic_number"] in {6, 7, 8})

    for pair in pairs:
        if len(selected) >= PILOT_PAIR_LIMIT:
            break
        if pair not in selected:
            selected.append(pair)

    return selected[:PILOT_PAIR_LIMIT]


pilot_pairs = select_pilot_pairs(registered_pairs)
selected_pairs = registered_pairs if RUN_FULL_BATCH else pilot_pairs

display(
    pd.DataFrame(
        [
            {
                "pair_key": p["pair_key"],
                "topic": p["topic_name"],
                "subtopic": p["subtopic_name"],
                "language": p["programming_language"],
                "mode": "batch" if RUN_FULL_BATCH else "pilot",
            }
            for p in selected_pairs
        ]
    )
)

print(f"Pairs selected: {len(selected_pairs)}")


,pair_key,topic,subtopic,language,mode
0,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,Fundamentals of Algorithms,Representing Algorithms,Python,batch
1,AQA_GCSE_CS_PMT_T1_1_2_PYTHON,Fundamentals of Algorithms,Efficiency of Algorithms,Python,batch
2,AQA_GCSE_CS_PMT_T1_1_3_PYTHON,Fundamentals of Algorithms,Searching Algorithms,Python,batch
3,AQA_GCSE_CS_PMT_T1_1_4_PYTHON,Fundamentals of Algorithms,Sorting Algorithms,Python,batch
4,AQA_GCSE_CS_PMT_T2_2_01_PYTHON,Programming,Data Types,Python,batch
5,AQA_GCSE_CS_PMT_T2_2_02_PYTHON,Programming,Programming Concepts,Python,batch
6,AQA_GCSE_CS_PMT_T2_2_03_PYTHON,Programming,Arithmetic Operations,Python,batch
7,AQA_GCSE_CS_PMT_T2_2_04_PYTHON,Programming,Relational Operations,Python,batch
8,AQA_GCSE_CS_PMT_T2_2_05_PYTHON,Programming,Boolean Operations,Python,batch
9,AQA_GCSE_CS_PMT_T2_2_06_PYTHON,Programming,Data Structures,Python,batch


Pairs selected: 35


## 7. Shared normalization, identifiers, hashes, and specification detection


In [49]:
def normalize_text(value: str) -> str:
    replacements = {
        "\u00a0": " ",
        "\uf0df": "←",
        "": "←",
        "–": "-",
        "—": "-",
    }
    for source, target in replacements.items():
        value = value.replace(source, target)
    return re.sub(r"[ \t]+", " ", value).strip()


def normalize_question_number(value: str) -> str:
    value = re.sub(r"\s+", "", value).replace("·", ".")
    parts = value.split(".")
    normalized: list[str] = []

    for part in parts:
        normalized.append(str(int(part)) if part.isdigit() else part.lower())

    return ".".join(normalized)


def split_question_number(value: str) -> tuple[str, str | None]:
    main, *parts = value.split(".")
    return main, parts[0] if parts else None


def safe_identifier_part(value: str) -> str:
    value = re.sub(r"[^A-Za-z0-9]+", "_", value)
    return re.sub(r"_+", "_", value).strip("_").upper()


def calculate_hash(payload: dict[str, Any]) -> str:
    encoded = json.dumps(
        payload,
        ensure_ascii=False,
        sort_keys=True,
        default=str,
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def make_json_safe(value: Any) -> Any:
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, dict):
        return {str(k): make_json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [make_json_safe(v) for v in value]
    if isinstance(value, (Path, uuid.UUID)):
        return str(value)
    if isinstance(value, datetime):
        return value.isoformat()
    if hasattr(value, "item"):
        try:
            return make_json_safe(value.item())
        except (TypeError, ValueError):
            pass
    return str(value)


def determine_specification_scope(
    page_numbers: list[int],
    page_metadata: dict[int, dict[str, Any]],
) -> str:
    has_8520 = any(
        page_metadata.get(page, {}).get("contains_8520_marker", False)
        for page in page_numbers
    )
    has_8525 = any(
        page_metadata.get(page, {}).get("contains_8525_marker", False)
        for page in page_numbers
    )

    if has_8520 and has_8525:
        return "mixed_8520_and_8525_markers"
    if has_8520:
        return "8520_marker_detected"
    if has_8525:
        return "8525_marker_detected"
    return "source_not_explicit_in_extracted_text"


## 8. Coordinate-aware topical Question Paper parser


### Parser v1.1 correction

"
            "- repeated continuation-page question headers are merged;
"
            "- generic words such as `algorithm` no longer automatically imply code;
"
            "- MS entries are detected against the expected QP sequence;
"
            "- total marks may be read from the total-marks column, AO mark sums, "
            "guidance text, or a review-flagged QP fallback.
"
            

In [50]:
QUESTION_MARKER_PATTERN = re.compile(
    r"^\s*(?P<tens>\d)\s+(?P<ones>\d)"
    r"(?:\s*\.\s*(?P<sub>\d+))?"
    r"(?P<remainder>.*)$"
)

MARK_LABEL_PATTERN = re.compile(
    r"\[\s*(\d+)\s+marks?\s*\]",
    re.IGNORECASE,
)

VISUAL_REFERENCE_PATTERN = re.compile(
    r"\b(figure|diagram|flowchart|graph|image|bitmap|"
    r"trace table|table\s+\d+|table below|huffman tree)\b",
    re.IGNORECASE,
)

CODE_REFERENCE_PATTERN = re.compile(
    r"\b("
    r"pseudo-?code|python\s+program|python\s+code|"
    r"write\s+a\s+program|write\s+code|"
    r"subroutine|procedure|function|"
    r"while\s+loop|for\s+loop|"
    r"sql\s+query|database\s+query|"
    r"algorithm\s+(?:shown|below)|"
    r"represented\s+using\s+pseudo-?code|"
    r"following\s+(?:code|program|algorithm)"
    r")\b"
    r"|(?:←|<-|==|!=|<=|>=)"
    r"|\b(?:IF|ELSE|ENDIF|WHILE|ENDWHILE|"
    r"FOR|ENDFOR|INPUT|OUTPUT|RETURN)\b",
    re.IGNORECASE,
)

PMT_SPEC_FOOTER_PATTERN = re.compile(
    r"^\*.*(?:8520|8525).*\*$",
    re.IGNORECASE,
)


def parse_question_marker_prefix(
    text: str,
    x0: float,
    font_sizes: tuple[float, ...],
) -> tuple[str | None, str]:
    match = QUESTION_MARKER_PATTERN.match(text)

    if match is None or x0 >= 145:
        return None, ""

    max_size = max(font_sizes or (0.0,))
    if not 7.0 <= max_size <= 16.0:
        return None, ""

    sub = match.group("sub")
    remainder = normalize_text(match.group("remainder") or "")

    # Avoid interpreting binary/numeric answer content as a question marker.
    if sub is None and remainder and re.fullmatch(r"[\d\s]+", remainder):
        return None, ""

    marker = f"{match.group('tens')}{match.group('ones')}"
    if sub:
        marker += f".{sub}"

    return marker, remainder


def should_drop_qp_fragment(
    text: str,
    x0: float,
    y0: float,
    page_width: float,
    page_height: float,
    subtopic_header: str,
) -> bool:
    lower = text.lower().strip()
    known_header = normalize_text(subtopic_header).lower()

    if "physicsandmathstutor.com" in lower:
        return True
    if y0 < 90 and known_header and lower == known_header:
        return True

    # Clipped AQA right-margin text.
    if x0 > page_width * 0.72 and lower.startswith(("do not w", "outside", "box")):
        return True
    if y0 < 180 and lower.startswith("do not w"):
        return True
    if (
        y0 < 180
        and x0 > page_width * 0.65
        and lower.startswith(("outside", "box"))
    ):
        return True

    exact_noise = {
        "do not write",
        "outside the",
        "outside",
        "outside t",
        "box",
        "outsidebox t",
        "do not write outside the box",
        "answer all questions.",
        "answer all questions in the spaces provided.",
        "answer all questions in the spaces provided",
        "turn over for the next question",
        "do not write on this page",
        "answer in the spaces provided",
        "blank page",
    }
    if lower in exact_noise:
        return True

    # Only remove the centred page number at the extreme top.
    # Do not remove left-column question digits such as 0 1 . 2.
    if (
        y0 < 25
        and re.fullmatch(r"\d+", text)
        and page_width * 0.35 < x0 < page_width * 0.65
    ):
        return True

    # Keep the marker in page metadata, but remove it from question text.
    if y0 > page_height * 0.86 and PMT_SPEC_FOOTER_PATTERN.fullmatch(text):
        return True

    if (
        y0 > page_height * 0.88
        and (
            re.fullmatch(r"\*?\d+\*?", text)
            or lower.startswith("turn over")
        )
    ):
        return True

    if re.fullmatch(r"\d+", text) and x0 > page_width * 0.60:
        return True
    if re.match(r"^Question \d+ continues", text, re.IGNORECASE):
        return True
    if lower == "answer" and x0 > page_width * 0.25:
        return True

    return False


def merge_page_fragments(
    fragments: list[dict[str, Any]],
    y_tolerance: float = 2.5,
) -> list[dict[str, Any]]:
    fragments = sorted(fragments, key=lambda item: (item["y0"], item["x0"]))
    grouped: list[list[dict[str, Any]]] = []

    for fragment in fragments:
        if not grouped:
            grouped.append([fragment])
            continue

        current = grouped[-1]
        row_y = sum(item["y0"] for item in current) / len(current)

        if abs(fragment["y0"] - row_y) <= y_tolerance:
            current.append(fragment)
        else:
            grouped.append([fragment])

    rows: list[dict[str, Any]] = []

    for group in grouped:
        group = sorted(group, key=lambda item: item["x0"])
        pieces: list[str] = []

        for fragment in group:
            fragment_text = fragment["text"]
            if pieces and re.match(r"^[.,:;)\]]", fragment_text):
                pieces[-1] += fragment_text
            else:
                pieces.append(fragment_text)

        rows.append(
            {
                "text": normalize_text(" ".join(pieces)),
                "x0": min(item["x0"] for item in group),
                "y0": min(item["y0"] for item in group),
                "x1": max(item["x1"] for item in group),
                "y1": max(item["y1"] for item in group),
                "font_sizes": tuple(
                    size
                    for item in group
                    for size in item["font_sizes"]
                ),
            }
        )

    return rows


def extract_qp_lines(
    pdf_path: Path,
    subtopic_code: str,
    subtopic_name: str,
) -> tuple[list[QuestionLine], dict[int, dict[str, Any]], int]:
    extracted: list[QuestionLine] = []
    page_metadata: dict[int, dict[str, Any]] = {}
    subtopic_header = f"{subtopic_code} {subtopic_name}"

    with fitz.open(pdf_path) as document:
        page_count = document.page_count

        for page_index, page in enumerate(document):
            page_number = page_index + 1
            page_dict = page.get_text("dict", sort=True)
            page_text = page.get_text("text", sort=True)

            meaningful_images: list[tuple[float, float, float, float]] = []

            for block in page_dict.get("blocks", []):
                if block.get("type") != 1:
                    continue

                x0, y0, x1, y1 = map(float, block["bbox"])
                area = max(0.0, x1 - x0) * max(0.0, y1 - y0)

                if area > 2500 and y0 > 80 and y1 < page.rect.height - 60:
                    meaningful_images.append((x0, y0, x1, y1))

            page_metadata[page_number] = {
                "width": float(page.rect.width),
                "height": float(page.rect.height),
                "meaningful_image_count": len(meaningful_images),
                "image_bboxes": meaningful_images,
                "contains_8520_marker": bool(
                    re.search(r"8520(?:/|\b)", page_text)
                ),
                "contains_8525_marker": bool(
                    re.search(r"8525(?:/|\b)", page_text)
                ),
            }

            fragments: list[dict[str, Any]] = []

            for block in page_dict.get("blocks", []):
                if block.get("type") != 0:
                    continue

                for line in block.get("lines", []):
                    spans = line.get("spans", [])
                    raw_text = "".join(
                        str(span.get("text", ""))
                        for span in spans
                    )
                    text = normalize_text(raw_text)

                    if not text:
                        continue

                    x0, y0, x1, y1 = map(float, line["bbox"])

                    if should_drop_qp_fragment(
                        text,
                        x0,
                        y0,
                        float(page.rect.width),
                        float(page.rect.height),
                        subtopic_header,
                    ):
                        continue

                    fragments.append(
                        {
                            "text": text,
                            "x0": x0,
                            "y0": y0,
                            "x1": x1,
                            "y1": y1,
                            "font_sizes": tuple(
                                float(span.get("size", 0.0))
                                for span in spans
                            ),
                        }
                    )

            for row in merge_page_fragments(fragments):
                marker, remainder = parse_question_marker_prefix(
                    row["text"],
                    row["x0"],
                    row["font_sizes"],
                )

                extracted.append(
                    QuestionLine(
                        page_number=page_number,
                        text=row["text"],
                        x0=row["x0"],
                        y0=row["y0"],
                        x1=row["x1"],
                        y1=row["y1"],
                        font_sizes=row["font_sizes"],
                        question_marker=marker,
                        marker_remainder=remainder,
                    )
                )

    return extracted, page_metadata, page_count


def clean_question_text(value: str) -> str:
    end_patterns = (
        re.compile(r"\n?END OF QUESTIONS.*", re.IGNORECASE | re.DOTALL),
        re.compile(r"\n?Copyright information.*", re.IGNORECASE | re.DOTALL),
        re.compile(
            r"\n?There are no questions printed on this page.*",
            re.IGNORECASE | re.DOTALL,
        ),
        re.compile(r"\n?Additional page.*", re.IGNORECASE | re.DOTALL),
    )

    for pattern in end_patterns:
        value = pattern.split(value, maxsplit=1)[0]

    return re.sub(r"\n{3,}", "\n\n", value).strip()


In [51]:
def merge_qp_continuation_segments(
    segments: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """
    PMT packs sometimes repeat the same original question number at the
    top of a continuation page. PyMuPDF then sees the continuation header
    as another question marker.

    Merge only the safe case:
    - same exact normalized question number;
    - first segment has no mark label;
    - second segment has a mark label;
    - pages are adjacent/overlapping.

    Genuine repeated questions from different papers normally both carry
    their own marks and therefore remain separate occurrences.
    """
    merged: list[dict[str, Any]] = []
    index = 0

    while index < len(segments):
        current = dict(segments[index])

        if index + 1 < len(segments):
            following = segments[index + 1]

            same_number = (
                normalize_question_number(
                    current["question_number"]
                )
                == normalize_question_number(
                    following["question_number"]
                )
            )

            adjacent_pages = (
                following["page_start"]
                <= current["page_end"] + 1
            )

            continuation_case = (
                same_number
                and current["marks"] is None
                and following["marks"] is not None
                and adjacent_pages
            )

            if continuation_case:
                second_question_text = re.sub(
                    r"^\s*[.\-–—:]+\s*",
                    "",
                    following["question_text"],
                )

                second_raw_text = re.sub(
                    r"^\s*[.\-–—:]+\s*",
                    "",
                    following["raw_text"],
                )

                current["question_text"] = (
                    "\n\n".join(
                        value
                        for value in (
                            current["question_text"],
                            second_question_text,
                        )
                        if value
                    )
                )

                current["raw_text"] = (
                    "\n\n".join(
                        value
                        for value in (
                            current["raw_text"],
                            second_raw_text,
                        )
                        if value
                    )
                )

                current["marks"] = following["marks"]
                current["mark_labels"] = [
                    *current["mark_labels"],
                    *following["mark_labels"],
                ]
                current["page_end"] = max(
                    current["page_end"],
                    following["page_end"],
                )
                current["visual_pages"] = sorted(
                    set(current["visual_pages"])
                    | set(following["visual_pages"])
                )

                scopes = {
                    current["specification_scope"],
                    following["specification_scope"],
                }

                has_8520 = (
                    "8520_marker_detected" in scopes
                    or "mixed_8520_and_8525_markers" in scopes
                )
                has_8525 = (
                    "8525_marker_detected" in scopes
                    or "mixed_8520_and_8525_markers" in scopes
                )

                if has_8520 and has_8525:
                    current["specification_scope"] = (
                        "mixed_8520_and_8525_markers"
                    )
                elif has_8520:
                    current["specification_scope"] = (
                        "8520_marker_detected"
                    )
                elif has_8525:
                    current["specification_scope"] = (
                        "8525_marker_detected"
                    )

                current["continuation_merged"] = True
                index += 2
                merged.append(current)
                continue

        current["continuation_merged"] = False
        merged.append(current)
        index += 1

    return merged


def parse_topical_qp(
    source: dict[str, Any],
) -> tuple[list[ParsedQuestion], list[dict[str, Any]]]:
    lines, page_metadata, page_count = extract_qp_lines(
        Path(source["question_path"]),
        str(source["subtopic_code"]),
        source["subtopic_name"],
    )

    marker_indexes = [
        index
        for index, line in enumerate(lines)
        if line.question_marker is not None
    ]

    if not marker_indexes:
        return [], [
            {
                "stage": "question_parsing",
                "severity": "critical",
                "issue_type": "no_question_markers_detected",
                "description": "No question markers were detected in the QP.",
                "document_id": source["question_document_id"],
                "details": {"path": str(source["question_path"])},
            }
        ]

    preliminary: list[dict[str, Any]] = []

    for marker_position, line_index in enumerate(marker_indexes):
        marker_line = lines[line_index]

        next_index = (
            marker_indexes[marker_position + 1]
            if marker_position + 1 < len(marker_indexes)
            else len(lines)
        )

        next_marker_page = (
            lines[next_index].page_number
            if next_index < len(lines)
            else page_count + 1
        )

        body_lines = lines[line_index + 1 : next_index]
        body_parts: list[str] = []

        if marker_line.marker_remainder:
            body_parts.append(marker_line.marker_remainder)

        body_parts.extend(line.text for line in body_lines)

        raw_text = clean_question_text(
            "\n".join(body_parts)
        )

        mark_labels = [
            int(value)
            for value in MARK_LABEL_PATTERN.findall(raw_text)
        ]

        marks = mark_labels[-1] if mark_labels else None

        question_text = clean_question_text(
            MARK_LABEL_PATTERN.sub("", raw_text)
        )

        body_pages = [
            line.page_number
            for line in body_lines
        ]

        page_start = marker_line.page_number
        page_end = max([page_start, *body_pages])

        if next_marker_page - page_start > 1:
            page_end = max(
                page_end,
                next_marker_page - 1,
            )

        if next_index == len(lines):
            page_end = max(page_end, page_count)

        source_pages = list(
            range(page_start, page_end + 1)
        )

        visual_pages = [
            page
            for page in source_pages
            if page_metadata.get(page, {}).get(
                "meaningful_image_count",
                0,
            )
            > 0
        ]

        preliminary.append(
            {
                "question_number": marker_line.question_marker,
                "raw_text": raw_text,
                "question_text": question_text,
                "marks": marks,
                "mark_labels": mark_labels,
                "page_start": page_start,
                "page_end": page_end,
                "visual_pages": visual_pages,
                "specification_scope": (
                    determine_specification_scope(
                        source_pages,
                        page_metadata,
                    )
                ),
            }
        )

    preliminary = merge_qp_continuation_segments(
        preliminary
    )

    # Reject rows from tables/data that resemble question numbers.
    # Examples found in the supplied cache:
    # - RLE data beginning with 60
    # - SQL table rows beginning with 11, 21, 33, 42
    discarded_data_segments: list[
        dict[str, Any]
    ] = []

    filtered_preliminary: list[
        dict[str, Any]
    ] = []

    for item in preliminary:
        normalized_candidate = (
            normalize_question_number(
                item["question_number"]
            )
        )

        candidate_main = (
            normalized_candidate.split(
                ".",
                maxsplit=1,
            )[0]
        )

        main_value = (
            int(candidate_main)
            if candidate_main.isdigit()
            else 0
        )

        question_body = (
            item["question_text"]
            .strip()
        )

        suspicious_data_row = (
            item["marks"] is None
            and (
                main_value > 20
                or bool(
                    re.match(
                        r"^\d{4}-\d{2}-\d{2}\b",
                        question_body,
                    )
                )
                or bool(
                    re.match(
                        r"^\d+(?:\s+\d+){2,}\b",
                        question_body,
                    )
                )
            )
        )

        if suspicious_data_row:
            discarded_data_segments.append(
                {
                    "question_number": (
                        item[
                            "question_number"
                        ]
                    ),
                    "page_start": (
                        item["page_start"]
                    ),
                    "text_preview": (
                        question_body[:160]
                    ),
                }
            )
            continue

        filtered_preliminary.append(
            item
        )

    preliminary = (
        filtered_preliminary
    )

    detected_numbers = [
        item["question_number"]
        for item in preliminary
    ]

    for index, item in enumerate(preliminary):
        normalized = normalize_question_number(
            item["question_number"]
        )

        main_number, part_number = (
            split_question_number(normalized)
        )

        later_child_exists = any(
            normalize_question_number(number).startswith(
                f"{main_number}."
            )
            for number in detected_numbers[index + 1 :]
        )

        is_context = (
            part_number is None
            and item["marks"] is None
            and later_child_exists
        )

        item["normalized_question_number"] = normalized
        item["main_question_number"] = main_number
        item["part_number"] = part_number
        item["record_type"] = (
            "context"
            if is_context
            else "scored_item"
        )
        item["parent_question_number"] = (
            main_number
            if part_number is not None
            else None
        )

    context_by_main = {
        item["normalized_question_number"]: item["question_text"]
        for item in preliminary
        if item["record_type"] == "context"
    }

    occurrences: Counter[str] = Counter()
    parsed: list[ParsedQuestion] = []
    issues: list[dict[str, Any]] = []

    if discarded_data_segments:
        issues.append(
            {
                "stage": (
                    "question_validation"
                ),
                "severity": "info",
                "issue_type": (
                    "non_question_data_rows_discarded"
                ),
                "description": (
                    "Numeric/date table rows that "
                    "resembled question markers were "
                    "discarded."
                ),
                "document_id": source[
                    "question_document_id"
                ],
                "details": {
                    "segments": (
                        discarded_data_segments
                    ),
                },
            }
        )

    for sequence_index, item in enumerate(
        preliminary,
        start=1,
    ):
        normalized = item[
            "normalized_question_number"
        ]

        occurrences[normalized] += 1
        occurrence_index = occurrences[normalized]

        context_text = (
            context_by_main.get(
                item["main_question_number"],
                "",
            )
            if item["record_type"] == "scored_item"
            else ""
        )

        searchable_body = "\n\n".join(
            value
            for value in (
                context_text,
                item["question_text"],
            )
            if value
        ).strip()

        search_text = (
            f"Topic: {source['topic_name']}\n"
            f"Subtopic: {source['subtopic_name']}\n\n"
            f"{searchable_body}"
        ).strip()

        has_visual = bool(
            item["visual_pages"]
            or VISUAL_REFERENCE_PATTERN.search(
                searchable_body
            )
        )

        has_code = bool(
            CODE_REFERENCE_PATTERN.search(
                searchable_body
            )
        )

        warnings: list[str] = []

        if item.get("continuation_merged"):
            warnings.append(
                "continuation_page_merged"
            )

        if (
            item["record_type"] == "scored_item"
            and item["marks"] is None
        ):
            warnings.append("missing_marks")

        if (
            item["record_type"] == "scored_item"
            and len(item["question_text"]) < 12
        ):
            warnings.append(
                "very_short_question_text"
            )

        if len(item["mark_labels"]) > 1:
            warnings.append(
                "multiple_mark_labels"
            )

        if has_visual:
            warnings.append(
                "visual_asset_review_recommended"
            )

        is_legacy = (
            item["specification_scope"]
            == "8520_marker_detected"
        )

        if is_legacy:
            warnings.append(
                "legacy_8520_question"
            )

        critical_warnings = {
            "missing_marks",
            "very_short_question_text",
            "multiple_mark_labels",
            "legacy_8520_question",
        }

        review_status = (
            "needs_review"
            if any(
                warning in critical_warnings
                for warning in warnings
            )
            else "auto_valid"
        )

        internal_key = (
            f"{normalized}#{occurrence_index}"
        )

        number_uid = safe_identifier_part(
            normalized
        )

        question_uid = (
            f"{source['pair_key']}_"
            f"QP_Q{number_uid}_"
            f"O{occurrence_index:02d}"
        )

        parsed.append(
            ParsedQuestion(
                internal_key=internal_key,
                question_uid=question_uid,
                pair_key=source["pair_key"],
                topic_id=source["topic_id"],
                question_document_id=(
                    source["question_document_id"]
                ),
                sequence_index=sequence_index,
                question_number=item[
                    "question_number"
                ],
                normalized_question_number=(
                    normalized
                ),
                occurrence_index=(
                    occurrence_index
                ),
                main_question_number=item[
                    "main_question_number"
                ],
                part_number=item["part_number"],
                parent_question_number=item[
                    "parent_question_number"
                ],
                record_type=item["record_type"],
                question_text=item[
                    "question_text"
                ],
                context_text=context_text,
                search_text=search_text,
                raw_extracted_text=item[
                    "raw_text"
                ],
                marks=item["marks"],
                page_start=item["page_start"],
                page_end=item["page_end"],
                has_visual=has_visual,
                visual_page_numbers=item[
                    "visual_pages"
                ],
                has_code=has_code,
                specification_scope=item[
                    "specification_scope"
                ],
                is_legacy=is_legacy,
                parse_warnings=warnings,
                review_status=review_status,
                content_hash=calculate_hash(
                    {
                        "pair_key": source[
                            "pair_key"
                        ],
                        "number": normalized,
                        "occurrence": (
                            occurrence_index
                        ),
                        "question_text": item[
                            "question_text"
                        ],
                        "context_text": (
                            context_text
                        ),
                        "marks": item["marks"],
                    }
                ),
            )
        )

    scored_items = [
        question
        for question in parsed
        if question.record_type == "scored_item"
    ]

    missing_marks = [
        question.internal_key
        for question in scored_items
        if question.marks is None
    ]

    if not scored_items:
        issues.append(
            {
                "stage": "question_validation",
                "severity": "critical",
                "issue_type": "no_scored_questions",
                "description": "No scored questions were parsed.",
                "document_id": source[
                    "question_document_id"
                ],
                "details": {},
            }
        )

    if missing_marks:
        issues.append(
            {
                "stage": "question_validation",
                "severity": "critical",
                "issue_type": (
                    "scored_questions_missing_marks"
                ),
                "description": (
                    "One or more scored questions "
                    "have no mark value."
                ),
                "document_id": source[
                    "question_document_id"
                ],
                "details": {
                    "question_keys": missing_marks
                },
            }
        )

    duplicate_counts = {
        number: count
        for number, count in Counter(
            item["normalized_question_number"]
            for item in preliminary
        ).items()
        if count > 1
    }

    if duplicate_counts:
        issues.append(
            {
                "stage": "question_validation",
                "severity": "info",
                "issue_type": (
                    "repeated_question_numbers_supported"
                ),
                "description": (
                    "Repeated question numbers were "
                    "retained using occurrence indexes."
                ),
                "document_id": source[
                    "question_document_id"
                ],
                "details": {
                    "counts": duplicate_counts
                },
            }
        )

    return parsed, issues


## 9. Coordinate-aware topical Mark Scheme parser


In [52]:
def group_words_into_rows(
    words: list[tuple],
    y_tolerance: float = 3.0,
) -> list[dict[str, Any]]:
    sorted_words = sorted(
        words,
        key=lambda word: (
            (
                float(word[1])
                + float(word[3])
            )
            / 2,
            float(word[0]),
        ),
    )

    grouped: list[list[tuple]] = []

    for word in sorted_words:
        centre_y = (
            float(word[1])
            + float(word[3])
        ) / 2

        if not grouped:
            grouped.append([word])
            continue

        current = grouped[-1]

        current_centre = sum(
            (
                float(item[1])
                + float(item[3])
            )
            / 2
            for item in current
        ) / len(current)

        if abs(
            centre_y
            - current_centre
        ) <= y_tolerance:
            current.append(word)
        else:
            grouped.append([word])

    rows: list[dict[str, Any]] = []

    for group in grouped:
        group = sorted(
            group,
            key=lambda word: float(word[0]),
        )

        rows.append(
            {
                "y0": min(
                    float(word[1])
                    for word in group
                ),
                "y1": max(
                    float(word[3])
                    for word in group
                ),
                "words": group,
                "text": normalize_text(
                    " ".join(
                        str(word[4])
                        for word in group
                    )
                ),
            }
        )

    return rows


def extract_ms_rows(
    document: fitz.Document,
) -> tuple[
    dict[int, list[MarkSchemeRow]],
    dict[int, dict[str, Any]],
]:
    rows_by_page: dict[
        int,
        list[MarkSchemeRow],
    ] = {}

    page_metadata: dict[
        int,
        dict[str, Any],
    ] = {}

    for page_index, page in enumerate(
        document
    ):
        page_number = page_index + 1

        raw_rows = group_words_into_rows(
            page.get_text(
                "words",
                sort=True,
            )
        )

        page_text = page.get_text(
            "text",
            sort=True,
        )

        rows_by_page[page_number] = [
            MarkSchemeRow(
                page_number=page_number,
                row_index=row_index,
                y0=row["y0"],
                y1=row["y1"],
                text=row["text"],
                words=row["words"],
            )
            for row_index, row in enumerate(
                raw_rows
            )
        ]

        page_metadata[page_number] = {
            "width": float(
                page.rect.width
            ),
            "height": float(
                page.rect.height
            ),
            "contains_8520_marker": bool(
                re.search(
                    r"8520(?:/|\b)",
                    page_text,
                )
            ),
            "contains_8525_marker": bool(
                re.search(
                    r"8525(?:/|\b)",
                    page_text,
                )
            ),
        }

    return rows_by_page, page_metadata


def is_ms_header_or_footer(
    row: MarkSchemeRow,
    page_height: float,
) -> bool:
    lower = normalize_text(
        row.text
    ).lower()

    centre_y = (
        row.y0
        + row.y1
    ) / 2

    if centre_y < 10:
        return True

    if centre_y > (
        page_height - 55
    ):
        return True

    if (
        "physicsandmathstutor.com"
        in lower
    ):
        return True

    tokens = set(
        lower.split()
    )

    if (
        "marking" in tokens
        and "guidance" in tokens
        and (
            "question" in tokens
            or "part" in tokens
            or "total" in tokens
        )
    ):
        return True

    if lower.startswith(
        "mark scheme - gcse computer science"
    ):
        return True

    return False


def detect_ms_candidate_starts(
    document: fitz.Document,
    rows_by_page: dict[
        int,
        list[MarkSchemeRow],
    ],
) -> list[MarkSchemeStart]:
    """
    Detect genuine MS entry starts.

    Across the supplied PMT cache, a real entry start:
    - contains a 1–2 digit question number in the far-left question column;
    - contains the word mark/marks on the same row;
    - may or may not contain a separate part number.
    """
    candidates: list[
        MarkSchemeStart
    ] = []

    for page_number, rows in (
        rows_by_page.items()
    ):
        page = document[
            page_number - 1
        ]

        page_height = float(
            page.rect.height
        )

        for row in rows:
            centre_y = (
                row.y0
                + row.y1
            ) / 2

            if (
                centre_y < 10
                or centre_y
                > page_height - 55
            ):
                continue

            if not re.search(
                r"\bmarks?\b",
                row.text,
                re.IGNORECASE,
            ):
                continue

            integer_words = [
                word
                for word in row.words
                if re.fullmatch(
                    r"\d{1,2}",
                    str(word[4]),
                )
            ]

            question_words = [
                word
                for word in integer_words
                if (
                    20
                    <= float(word[0])
                    <= 95
                )
            ]

            if not question_words:
                continue

            question_word = min(
                question_words,
                key=lambda word: (
                    float(word[0])
                ),
            )

            main_number = str(
                int(question_word[4])
            )

            part_words = [
                word
                for word in integer_words
                if (
                    float(word[0])
                    > float(
                        question_word[2]
                    )
                    + 2
                    and float(word[0])
                    <= 135
                )
            ]

            part_word = (
                min(
                    part_words,
                    key=lambda word: (
                        float(word[0])
                    ),
                )
                if part_words
                else None
            )

            part_number = (
                str(
                    int(part_word[4])
                )
                if part_word
                is not None
                else None
            )

            source_number = main_number

            variants = {
                main_number
            }

            if part_number is not None:
                variants.add(
                    f"{main_number}."
                    f"{part_number}"
                )

            candidates.append(
                MarkSchemeStart(
                    page_number=(
                        page_number
                    ),
                    row_index=(
                        row.row_index
                    ),
                    y=centre_y,
                    source_main_number=(
                        main_number
                    ),
                    source_part_number=(
                        part_number
                    ),
                    source_question_number=(
                        source_number
                    ),
                    source_number_variants=(
                        tuple(
                            sorted(variants)
                        )
                    ),
                    row_text=(
                        row.text
                    ),
                    row_words=(
                        row.words
                    ),
                    question_word=(
                        question_word
                    ),
                    part_word=(
                        part_word
                    ),
                )
            )

    return candidates


def infer_candidate_row_marks(
    candidate: MarkSchemeStart,
) -> int | None:
    far_right_values = [
        int(word[4])
        for word in candidate.row_words
        if (
            float(word[0])
            >= 500
            and re.fullmatch(
                r"\d{1,2}",
                str(word[4]),
            )
        )
    ]

    if far_right_values:
        return far_right_values[-1]

    ao_values = [
        int(value)
        for value in re.findall(
            r"\b(\d+)\s+marks?"
            r"\s+for\s+AO[123]\b",
            candidate.row_text,
            re.IGNORECASE,
        )
    ]

    if ao_values:
        return sum(ao_values)

    if re.search(
        r"\bMark is for AO[123]\b",
        candidate.row_text,
        re.IGNORECASE,
    ):
        return 1

    direct_match = re.search(
        r"\b(\d+)\s+marks?\b",
        candidate.row_text,
        re.IGNORECASE,
    )

    if direct_match:
        return int(
            direct_match.group(1)
        )

    return None


def score_question_candidate_match(
    question: ParsedQuestion,
    candidate: MarkSchemeStart,
) -> tuple[
    int,
    str,
]:
    expected_number = (
        question
        .normalized_question_number
    )

    expected_main = (
        expected_number.split(
            ".",
            maxsplit=1,
        )[0]
    )

    candidate_marks = (
        infer_candidate_row_marks(
            candidate
        )
    )

    if (
        expected_number
        in candidate
        .source_number_variants
    ):
        score = 12
        method = (
            "exact_number_occurrence"
        )

    elif (
        expected_main
        == candidate
        .source_main_number
    ):
        score = 6
        method = (
            "same_main_number_alignment"
        )

    else:
        score = 1
        method = (
            "sequence_alignment"
        )

    if (
        question.marks
        is not None
        and candidate_marks
        is not None
    ):
        if (
            question.marks
            == candidate_marks
        ):
            score += 5
        else:
            score -= 3

    return score, method


def align_questions_to_ms_candidates(
    questions: list[
        ParsedQuestion
    ],
    candidates: list[
        MarkSchemeStart
    ],
) -> tuple[
    list[dict[str, Any]],
    list[ParsedQuestion],
    list[MarkSchemeStart],
]:
    """
    Global sequence alignment.

    This avoids unsafe zipping while supporting:
    - exact numbering;
    - same-main numbering;
    - PMT QP/MS number differences;
    - extra MS candidate rows;
    - missing MS entries.
    """
    expected = [
        question
        for question in questions
        if question.record_type
        == "scored_item"
    ]

    question_count = len(
        expected
    )

    candidate_count = len(
        candidates
    )

    negative_infinity = (
        -10**9
    )

    scores = [
        [
            negative_infinity
            for _ in range(
                candidate_count + 1
            )
        ]
        for _ in range(
            question_count + 1
        )
    ]

    backtrack: list[
        list[
            tuple | None
        ]
    ] = [
        [
            None
            for _ in range(
                candidate_count + 1
            )
        ]
        for _ in range(
            question_count + 1
        )
    ]

    scores[0][0] = 0

    for question_index in range(
        question_count + 1
    ):
        for candidate_index in range(
            candidate_count + 1
        ):
            current_score = scores[
                question_index
            ][
                candidate_index
            ]

            if (
                current_score
                <= negative_infinity // 2
            ):
                continue

            if (
                question_index
                < question_count
            ):
                proposed = (
                    current_score - 5
                )

                if proposed > scores[
                    question_index + 1
                ][
                    candidate_index
                ]:
                    scores[
                        question_index + 1
                    ][
                        candidate_index
                    ] = proposed

                    backtrack[
                        question_index + 1
                    ][
                        candidate_index
                    ] = (
                        "missing_question_match",
                        question_index,
                        candidate_index,
                    )

            if (
                candidate_index
                < candidate_count
            ):
                proposed = (
                    current_score - 2
                )

                if proposed > scores[
                    question_index
                ][
                    candidate_index + 1
                ]:
                    scores[
                        question_index
                    ][
                        candidate_index + 1
                    ] = proposed

                    backtrack[
                        question_index
                    ][
                        candidate_index + 1
                    ] = (
                        "skip_candidate",
                        question_index,
                        candidate_index,
                    )

            if (
                question_index
                < question_count
                and candidate_index
                < candidate_count
            ):
                (
                    match_score,
                    method,
                ) = (
                    score_question_candidate_match(
                        expected[
                            question_index
                        ],
                        candidates[
                            candidate_index
                        ],
                    )
                )

                proposed = (
                    current_score
                    + match_score
                )

                if proposed > scores[
                    question_index + 1
                ][
                    candidate_index + 1
                ]:
                    scores[
                        question_index + 1
                    ][
                        candidate_index + 1
                    ] = proposed

                    backtrack[
                        question_index + 1
                    ][
                        candidate_index + 1
                    ] = (
                        "match",
                        question_index,
                        candidate_index,
                        match_score,
                        method,
                    )

    question_index = (
        question_count
    )

    candidate_index = (
        candidate_count
    )

    aligned: list[
        dict[str, Any]
    ] = []

    missing_question_indexes: list[
        int
    ] = []

    skipped_candidate_indexes: list[
        int
    ] = []

    while (
        question_index > 0
        or candidate_index > 0
    ):
        action = backtrack[
            question_index
        ][
            candidate_index
        ]

        if action is None:
            break

        action_type = action[0]

        if action_type == "match":
            (
                _,
                previous_question_index,
                previous_candidate_index,
                match_score,
                method,
            ) = action

            aligned.append(
                {
                    "question": expected[
                        previous_question_index
                    ],
                    "candidate": candidates[
                        previous_candidate_index
                    ],
                    "candidate_index": (
                        previous_candidate_index
                    ),
                    "alignment_score": (
                        match_score
                    ),
                    "alignment_method": (
                        method
                    ),
                }
            )

            question_index = (
                previous_question_index
            )

            candidate_index = (
                previous_candidate_index
            )

        elif (
            action_type
            == "missing_question_match"
        ):
            (
                _,
                previous_question_index,
                previous_candidate_index,
            ) = action

            missing_question_indexes.append(
                previous_question_index
            )

            question_index = (
                previous_question_index
            )

            candidate_index = (
                previous_candidate_index
            )

        else:
            (
                _,
                previous_question_index,
                previous_candidate_index,
            ) = action

            skipped_candidate_indexes.append(
                previous_candidate_index
            )

            question_index = (
                previous_question_index
            )

            candidate_index = (
                previous_candidate_index
            )

    aligned.reverse()

    missing_question_indexes.reverse()
    skipped_candidate_indexes.reverse()

    missing_questions = [
        expected[index]
        for index in (
            missing_question_indexes
        )
    ]

    skipped_candidates = [
        candidates[index]
        for index in (
            skipped_candidate_indexes
        )
    ]

    return (
        aligned,
        missing_questions,
        skipped_candidates,
    )


def same_pdf_word(
    first: tuple,
    second: tuple | None,
) -> bool:
    if second is None:
        return False

    return (
        abs(
            float(first[0])
            - float(second[0])
        )
        < 0.5
        and abs(
            float(first[1])
            - float(second[1])
        )
        < 0.5
        and str(first[4])
        == str(second[4])
    )


def extract_guidance_row(
    row: MarkSchemeRow,
    page_width: float,
    candidate: (
        MarkSchemeStart | None
    ),
    remove_part_word: bool,
) -> str:
    guidance_x_min = (
        page_width * 0.15
    )

    marks_x_min = (
        page_width * 0.82
    )

    selected: list[
        tuple
    ] = []

    for word in row.words:
        x0 = float(
            word[0]
        )

        if (
            x0 >= marks_x_min
            or x0 < guidance_x_min
        ):
            continue

        if candidate is not None:
            if same_pdf_word(
                word,
                candidate.question_word,
            ):
                continue

            if (
                remove_part_word
                and same_pdf_word(
                    word,
                    candidate.part_word,
                )
            ):
                continue

        selected.append(
            word
        )

    return normalize_text(
        " ".join(
            str(word[4])
            for word in sorted(
                selected,
                key=lambda item: (
                    float(item[0])
                ),
            )
        )
    )


def extract_total_mark_candidates(
    row: MarkSchemeRow,
    page_width: float,
) -> list[int]:
    marks_x_min = (
        page_width * 0.82
    )

    return [
        int(word[4])
        for word in row.words
        if (
            float(word[0])
            >= marks_x_min
            and re.fullmatch(
                r"\d{1,2}",
                str(word[4]),
            )
        )
    ]


def infer_entry_marks(
    guidance: str,
    total_mark_candidates: list[
        int
    ],
    question_marks: int | None,
) -> tuple[
    int | None,
    str,
]:
    if total_mark_candidates:
        return (
            total_mark_candidates[-1],
            "ms_total_marks_column",
        )

    ao_values = [
        int(value)
        for value in re.findall(
            r"\b(\d+)\s+marks?"
            r"\s+for\s+AO[123]\b",
            guidance,
            re.IGNORECASE,
        )
    ]

    if ao_values:
        return (
            sum(ao_values),
            "summed_ao_marks",
        )

    direct_values = [
        int(value)
        for value in re.findall(
            r"\b(\d+)\s+marks?\b",
            guidance,
            re.IGNORECASE,
        )
    ]

    if direct_values:
        candidate = max(
            direct_values
        )

        if (
            question_marks is None
            or candidate
            == question_marks
        ):
            return (
                candidate,
                "guidance_mark_phrase",
            )

    if re.search(
        r"\bMark is for AO[123]\b",
        guidance,
        re.IGNORECASE,
    ):
        return (
            1,
            "single_mark_ao_phrase",
        )

    return (
        question_marks,
        "qp_marks_fallback",
    )


def classify_guidance(
    guidance: str,
) -> dict[str, list[str]]:
    marking_points: list[str] = []
    acceptable_answers: list[str] = []
    rejected_answers: list[str] = []
    additional_guidance: list[str] = []

    lines = [
        normalize_text(line)
        for line in guidance.splitlines()
        if normalize_text(line)
    ]

    for line in lines:
        if re.match(
            r"^(A\.?|Allow\b|Accept\b)",
            line,
            re.IGNORECASE,
        ):
            acceptable_answers.append(
                line
            )

        elif re.match(
            r"^(R\.?|Reject\b)",
            line,
            re.IGNORECASE,
        ):
            rejected_answers.append(
                line
            )

        elif re.match(
            r"^(I\.?|Ignore\b|NE\b|DPT\b|"
            r"Note\b|If\b|Do not\b|"
            r"Examiner\b)",
            line,
            re.IGNORECASE,
        ):
            additional_guidance.append(
                line
            )

        elif re.search(
            r"\bAO[123]\b",
            line,
            re.IGNORECASE,
        ):
            continue

        else:
            marking_points.append(
                line
            )

    return {
        "marking_points": (
            marking_points
        ),
        "acceptable_answers": (
            acceptable_answers
        ),
        "rejected_answers": (
            rejected_answers
        ),
        "additional_guidance": (
            additional_guidance
        ),
    }


In [53]:
def parse_topical_ms(
    source: dict[str, Any],
    expected_questions: list[
        ParsedQuestion
    ],
) -> tuple[
    list[ParsedMarkSchemeEntry],
    list[dict[str, Any]],
]:
    pdf_path = Path(
        source["mark_scheme_path"]
    )

    issues: list[
        dict[str, Any]
    ] = []

    with fitz.open(
        pdf_path
    ) as document:
        (
            rows_by_page,
            page_metadata,
        ) = extract_ms_rows(
            document
        )

        candidates = (
            detect_ms_candidate_starts(
                document,
                rows_by_page,
            )
        )

        (
            aligned_records,
            missing_questions,
            skipped_candidates,
        ) = (
            align_questions_to_ms_candidates(
                expected_questions,
                candidates,
            )
        )

        parsed_entries: list[
            ParsedMarkSchemeEntry
        ] = []

        for sequence_index, alignment in enumerate(
            aligned_records,
            start=1,
        ):
            question = alignment[
                "question"
            ]

            candidate = alignment[
                "candidate"
            ]

            candidate_index = alignment[
                "candidate_index"
            ]

            alignment_score = alignment[
                "alignment_score"
            ]

            alignment_method = alignment[
                "alignment_method"
            ]

            next_candidate = (
                candidates[
                    candidate_index + 1
                ]
                if candidate_index + 1
                < len(candidates)
                else None
            )

            final_page = (
                next_candidate.page_number
                if next_candidate
                is not None
                else document.page_count
            )

            guidance_lines: list[
                str
            ] = []

            raw_lines: list[
                str
            ] = []

            total_mark_candidates: list[
                int
            ] = []

            page_end = (
                candidate.page_number
            )

            expected_has_part = (
                "."
                in question
                .normalized_question_number
            )

            candidate_exact_part = (
                expected_has_part
                and candidate
                .source_part_number
                is not None
                and (
                    question
                    .normalized_question_number
                    == (
                        f"{candidate.source_main_number}."
                        f"{candidate.source_part_number}"
                    )
                )
            )

            for page_number in range(
                candidate.page_number,
                final_page + 1,
            ):
                page_rows = (
                    rows_by_page[
                        page_number
                    ]
                )

                start_row_index = (
                    candidate.row_index
                    if page_number
                    == candidate.page_number
                    else 0
                )

                end_row_index = (
                    next_candidate.row_index
                    if (
                        next_candidate
                        is not None
                        and page_number
                        == next_candidate.page_number
                    )
                    else len(
                        page_rows
                    )
                )

                page = document[
                    page_number - 1
                ]

                page_width = float(
                    page.rect.width
                )

                page_height = float(
                    page.rect.height
                )

                for row_index in range(
                    start_row_index,
                    end_row_index,
                ):
                    row = page_rows[
                        row_index
                    ]

                    if is_ms_header_or_footer(
                        row,
                        page_height,
                    ):
                        continue

                    total_mark_candidates.extend(
                        extract_total_mark_candidates(
                            row,
                            page_width,
                        )
                    )

                    current_candidate = (
                        candidate
                        if (
                            page_number
                            == candidate.page_number
                            and row_index
                            == candidate.row_index
                        )
                        else None
                    )

                    line = extract_guidance_row(
                        row,
                        page_width,
                        current_candidate,
                        remove_part_word=(
                            candidate_exact_part
                        ),
                    )

                    if not line:
                        continue

                    if line.lower() in {
                        "marking guidance",
                        "part marking guidance",
                        "marks",
                        "total marks",
                    }:
                        continue

                    guidance_lines.append(
                        line
                    )

                    raw_lines.append(
                        row.text
                    )

                    page_end = (
                        page_number
                    )

            marking_guidance = (
                "\n".join(
                    guidance_lines
                ).strip()
            )

            (
                maximum_marks,
                marks_source,
            ) = infer_entry_marks(
                marking_guidance,
                total_mark_candidates,
                question.marks,
            )

            warnings: list[
                str
            ] = [
                (
                    "alignment_method="
                    f"{alignment_method}"
                ),
                (
                    "alignment_score="
                    f"{alignment_score}"
                ),
                (
                    "ms_source_main="
                    f"{candidate.source_main_number}"
                ),
            ]

            if (
                candidate
                .source_part_number
                is not None
            ):
                warnings.append(
                    (
                        "ms_source_part="
                        f"{candidate.source_part_number}"
                    )
                )

            if (
                alignment_method
                != "exact_number_occurrence"
            ):
                warnings.append(
                    "question_number_alignment_requires_review"
                )

            if not marking_guidance:
                warnings.append(
                    "empty_marking_guidance"
                )

            if maximum_marks is None:
                warnings.append(
                    "missing_mark_scheme_marks"
                )

                maximum_marks = (
                    question.marks
                    or 1
                )

            if (
                question.marks
                is not None
                and maximum_marks
                != question.marks
            ):
                warnings.append(
                    "ms_marks_differ_from_qp"
                )

            if (
                marks_source
                == "qp_marks_fallback"
            ):
                warnings.append(
                    "ms_marks_inferred_from_qp"
                )

            classified = (
                classify_guidance(
                    marking_guidance
                )
            )

            objectives = sorted(
                {
                    match.upper()
                    for match in re.findall(
                        r"\bAO[123]\b",
                        marking_guidance,
                        re.IGNORECASE,
                    )
                }
            )

            source_pages = list(
                range(
                    candidate.page_number,
                    page_end + 1,
                )
            )

            specification_scope = (
                determine_specification_scope(
                    source_pages,
                    page_metadata,
                )
            )

            is_legacy = (
                specification_scope
                == "8520_marker_detected"
            )

            if is_legacy:
                warnings.append(
                    "legacy_8520_mark_scheme_entry"
                )

            critical_warnings = {
                "empty_marking_guidance",
                "missing_mark_scheme_marks",
                "ms_marks_differ_from_qp",
                (
                    "question_number_alignment_"
                    "requires_review"
                ),
                (
                    "legacy_8520_"
                    "mark_scheme_entry"
                ),
            }

            review_status = (
                "needs_review"
                if any(
                    warning
                    in critical_warnings
                    for warning in warnings
                )
                else "auto_valid"
            )

            number_uid = (
                safe_identifier_part(
                    question
                    .normalized_question_number
                )
            )

            mark_scheme_uid = (
                f"{source['pair_key']}_"
                f"MS_Q{number_uid}_"
                f"O{question.occurrence_index:02d}"
            )

            parsed_entries.append(
                ParsedMarkSchemeEntry(
                    internal_key=(
                        question.internal_key
                    ),
                    mark_scheme_uid=(
                        mark_scheme_uid
                    ),
                    pair_key=source[
                        "pair_key"
                    ],
                    topic_id=source[
                        "topic_id"
                    ],
                    mark_scheme_document_id=(
                        source[
                            "mark_scheme_document_id"
                        ]
                    ),
                    sequence_index=(
                        sequence_index
                    ),
                    question_number=(
                        question.question_number
                    ),
                    normalized_question_number=(
                        question
                        .normalized_question_number
                    ),
                    occurrence_index=(
                        question
                        .occurrence_index
                    ),
                    main_question_number=(
                        question
                        .main_question_number
                    ),
                    part_number=(
                        question.part_number
                    ),
                    maximum_marks=(
                        int(maximum_marks)
                    ),
                    marking_guidance=(
                        marking_guidance
                    ),
                    marking_points=(
                        classified[
                            "marking_points"
                        ]
                    ),
                    acceptable_answers=(
                        classified[
                            "acceptable_answers"
                        ]
                    ),
                    rejected_answers=(
                        classified[
                            "rejected_answers"
                        ]
                    ),
                    additional_guidance=(
                        classified[
                            "additional_guidance"
                        ]
                    ),
                    assessment_objectives=(
                        objectives
                    ),
                    page_start=(
                        candidate.page_number
                    ),
                    page_end=(
                        page_end
                    ),
                    raw_extracted_text=(
                        "\n".join(
                            raw_lines
                        ).strip()
                    ),
                    specification_scope=(
                        specification_scope
                    ),
                    is_legacy=(
                        is_legacy
                    ),
                    parse_warnings=(
                        warnings
                    ),
                    review_status=(
                        review_status
                    ),
                    content_hash=(
                        calculate_hash(
                            {
                                "pair_key": (
                                    source[
                                        "pair_key"
                                    ]
                                ),
                                "question_key": (
                                    question
                                    .internal_key
                                ),
                                "maximum_marks": (
                                    int(
                                        maximum_marks
                                    )
                                ),
                                "marking_guidance": (
                                    marking_guidance
                                ),
                                "alignment_method": (
                                    alignment_method
                                ),
                                "alignment_score": (
                                    alignment_score
                                ),
                            }
                        )
                    ),
                )
            )

    if missing_questions:
        issues.append(
            {
                "stage": (
                    "mark_scheme_alignment"
                ),
                "severity": (
                    "critical"
                ),
                "issue_type": (
                    "questions_without_ms_candidate"
                ),
                "description": (
                    "One or more scored QP questions "
                    "could not be aligned to an MS entry."
                ),
                "document_id": source[
                    "mark_scheme_document_id"
                ],
                "details": {
                    "question_keys": [
                        question.internal_key
                        for question
                        in missing_questions
                    ],
                },
            }
        )

    if skipped_candidates:
        issues.append(
            {
                "stage": (
                    "mark_scheme_alignment"
                ),
                "severity": (
                    "warning"
                ),
                "issue_type": (
                    "unmatched_ms_candidates"
                ),
                "description": (
                    "One or more detected MS starts "
                    "were not aligned to a QP question."
                ),
                "document_id": source[
                    "mark_scheme_document_id"
                ],
                "details": {
                    "candidates": [
                        {
                            "page": (
                                candidate.page_number
                            ),
                            "row": (
                                candidate.row_index
                            ),
                            "main": (
                                candidate
                                .source_main_number
                            ),
                            "part": (
                                candidate
                                .source_part_number
                            ),
                            "text": (
                                candidate.row_text
                            ),
                        }
                        for candidate
                        in skipped_candidates
                    ],
                },
            }
        )

    empty_entries = [
        entry.internal_key
        for entry in parsed_entries
        if not entry.marking_guidance
    ]

    if empty_entries:
        issues.append(
            {
                "stage": (
                    "mark_scheme_validation"
                ),
                "severity": (
                    "critical"
                ),
                "issue_type": (
                    "empty_marking_guidance"
                ),
                "description": (
                    "One or more aligned entries "
                    "have no marking guidance."
                ),
                "document_id": source[
                    "mark_scheme_document_id"
                ],
                "details": {
                    "entry_keys": (
                        empty_entries
                    ),
                },
            }
        )

    return parsed_entries, issues


## 10. Deterministic QP ↔ MS linking

Primary key:

```text
normalized question number + occurrence index
```

Exact links are preferred. A sequence fallback is allowed only when the remaining
counts match exactly; fallback links always require human review.


In [54]:
def link_questions_to_ms(
    source: dict[str, Any],
    questions: list[ParsedQuestion],
    entries: list[ParsedMarkSchemeEntry],
) -> tuple[
    list[ProposedLink],
    list[dict[str, Any]],
]:
    scored_questions = [
        question
        for question in questions
        if question.record_type
        == "scored_item"
    ]

    question_by_key = {
        question.internal_key: (
            question
        )
        for question in scored_questions
    }

    entry_by_key = {
        entry.internal_key: (
            entry
        )
        for entry in entries
    }

    links: list[
        ProposedLink
    ] = []

    issues: list[
        dict[str, Any]
    ] = []

    common_keys = [
        question.internal_key
        for question in scored_questions
        if question.internal_key
        in entry_by_key
    ]

    for key in common_keys:
        question = (
            question_by_key[key]
        )

        entry = (
            entry_by_key[key]
        )

        alignment_method = (
            "exact_number_occurrence"
        )

        alignment_score = 17

        for warning in (
            entry.parse_warnings
        ):
            if warning.startswith(
                "alignment_method="
            ):
                alignment_method = (
                    warning.split(
                        "=",
                        maxsplit=1,
                    )[1]
                )

            elif warning.startswith(
                "alignment_score="
            ):
                try:
                    alignment_score = int(
                        warning.split(
                            "=",
                            maxsplit=1,
                        )[1]
                    )
                except ValueError:
                    pass

        marks_match = (
            question.marks
            == entry.maximum_marks
            if question.marks
            is not None
            else True
        )

        warnings: list[
            str
        ] = []

        if not marks_match:
            warnings.append(
                "marks_mismatch"
            )

        if (
            question.review_status
            == "needs_review"
        ):
            warnings.append(
                "question_needs_review"
            )

        if (
            entry.review_status
            == "needs_review"
        ):
            warnings.append(
                "mark_scheme_needs_review"
            )

        if (
            alignment_method
            != "exact_number_occurrence"
        ):
            warnings.append(
                "number_alignment_requires_review"
            )

        if (
            alignment_method
            == "exact_number_occurrence"
        ):
            confidence = (
                1.0
                if marks_match
                else 0.85
            )

        elif (
            alignment_method
            == "same_main_number_alignment"
        ):
            confidence = (
                0.85
                if marks_match
                else 0.65
            )

        else:
            confidence = (
                0.65
                if marks_match
                else 0.45
            )

        if alignment_score < 5:
            confidence = min(
                confidence,
                0.60,
            )

        links.append(
            ProposedLink(
                question_internal_key=(
                    question.internal_key
                ),
                mark_scheme_internal_key=(
                    entry.internal_key
                ),
                pair_key=source[
                    "pair_key"
                ],
                match_method=(
                    alignment_method
                ),
                match_confidence=(
                    confidence
                ),
                marks_match=(
                    marks_match
                ),
                validation_status=(
                    "auto_valid"
                    if not warnings
                    else "needs_review"
                ),
                validation_warnings=(
                    warnings
                ),
            )
        )

    linked_question_keys = {
        link.question_internal_key
        for link in links
    }

    linked_entry_keys = {
        link.mark_scheme_internal_key
        for link in links
    }

    unlinked_questions = [
        question.internal_key
        for question in scored_questions
        if question.internal_key
        not in linked_question_keys
    ]

    unlinked_entries = [
        entry.internal_key
        for entry in entries
        if entry.internal_key
        not in linked_entry_keys
    ]

    if unlinked_questions:
        issues.append(
            {
                "stage": (
                    "question_mark_scheme_linking"
                ),
                "severity": (
                    "critical"
                ),
                "issue_type": (
                    "unlinked_questions"
                ),
                "description": (
                    "One or more scored questions "
                    "have no linked MS entry."
                ),
                "document_id": source[
                    "question_document_id"
                ],
                "details": {
                    "question_keys": (
                        unlinked_questions
                    ),
                },
            }
        )

    if unlinked_entries:
        issues.append(
            {
                "stage": (
                    "question_mark_scheme_linking"
                ),
                "severity": (
                    "critical"
                ),
                "issue_type": (
                    "unlinked_mark_scheme_entries"
                ),
                "description": (
                    "One or more parsed MS entries "
                    "have no linked question."
                ),
                "document_id": source[
                    "mark_scheme_document_id"
                ],
                "details": {
                    "entry_keys": (
                        unlinked_entries
                    ),
                },
            }
        )

    mismatches = [
        {
            "question_key": (
                link
                .question_internal_key
            ),
            "mark_scheme_key": (
                link
                .mark_scheme_internal_key
            ),
        }
        for link in links
        if not link.marks_match
    ]

    if mismatches:
        issues.append(
            {
                "stage": (
                    "question_mark_scheme_linking"
                ),
                "severity": (
                    "critical"
                ),
                "issue_type": (
                    "linked_marks_mismatch"
                ),
                "description": (
                    "Linked QP and MS mark "
                    "allocations do not match."
                ),
                "document_id": source[
                    "question_document_id"
                ],
                "details": {
                    "links": (
                        mismatches
                    ),
                },
            }
        )

    review_alignments = [
        link.model_dump(
            mode="json"
        )
        for link in links
        if (
            link.match_method
            != "exact_number_occurrence"
        )
    ]

    if review_alignments:
        issues.append(
            {
                "stage": (
                    "question_mark_scheme_linking"
                ),
                "severity": (
                    "warning"
                ),
                "issue_type": (
                    "number_alignment_review_queue"
                ),
                "description": (
                    "Some QP/MS links were aligned "
                    "by main number or sequence and "
                    "require human confirmation."
                ),
                "document_id": source[
                    "question_document_id"
                ],
                "details": {
                    "links": (
                        review_alignments
                    ),
                },
            }
        )

    return links, issues


## 11. Parse and validate one complete QP/MS pair


In [55]:
def parse_pair(source: dict[str, Any]) -> PairParseResult:
    started = time.perf_counter()

    questions, question_issues = parse_topical_qp(source)
    entries, ms_issues = parse_topical_ms(
        source,
        questions,
    )
    links, link_issues = link_questions_to_ms(
        source,
        questions,
        entries,
    )

    issues = [
        *question_issues,
        *ms_issues,
        *link_issues,
    ]

    scored_count = sum(
        question.record_type == "scored_item"
        for question in questions
    )
    context_count = sum(
        question.record_type == "context"
        for question in questions
    )
    exact_count = sum(
        link.match_method == "exact_number_occurrence"
        for link in links
    )

    fallback_count = sum(
        link.match_method
        != "exact_number_occurrence"
        for link in links
    )

    linked_question_keys = {
        link.question_internal_key
        for link in links
    }
    linked_entry_keys = {
        link.mark_scheme_internal_key
        for link in links
    }

    unlinked_question_count = sum(
        question.record_type == "scored_item"
        and question.internal_key not in linked_question_keys
        for question in questions
    )
    unlinked_ms_count = sum(
        entry.internal_key not in linked_entry_keys
        for entry in entries
    )
    mismatch_count = sum(
        not link.marks_match
        for link in links
    )

    critical_issues = [
        issue
        for issue in issues
        if issue["severity"] == "critical"
    ]

    strict_passed = (
        not critical_issues
        and scored_count > 0
        and len(entries) > 0
        and unlinked_question_count == 0
        and unlinked_ms_count == 0
        and mismatch_count == 0
    )

    needs_review = bool(
        any(
            question.review_status == "needs_review"
            for question in questions
        )
        or any(
            entry.review_status == "needs_review"
            for entry in entries
        )
        or any(
            link.validation_status == "needs_review"
            for link in links
        )
        or any(
            issue["severity"] in {"warning", "critical"}
            for issue in issues
        )
    )

    return PairParseResult(
        pair_key=source["pair_key"],
        topic_id=source["topic_id"],
        topic_number=source["topic_number"],
        topic_name=source["topic_name"],
        subtopic_code=str(source["subtopic_code"]),
        subtopic_name=source["subtopic_name"],
        paper_code=source["paper_code"],
        programming_language=source["programming_language"],
        question_document_id=source["question_document_id"],
        mark_scheme_document_id=source["mark_scheme_document_id"],
        question_path=str(source["question_path"]),
        mark_scheme_path=str(source["mark_scheme_path"]),
        questions=questions,
        mark_scheme_entries=entries,
        links=links,
        issues=issues,
        scored_question_count=scored_count,
        context_count=context_count,
        mark_scheme_entry_count=len(entries),
        exact_link_count=exact_count,
        fallback_link_count=fallback_count,
        unlinked_question_count=unlinked_question_count,
        unlinked_mark_scheme_count=unlinked_ms_count,
        mark_mismatch_count=mismatch_count,
        strict_passed=strict_passed,
        needs_human_review=needs_review,
        processing_seconds=round(
            time.perf_counter() - started,
            4,
        ),
    )


## Final alignment policy

The complete supplied cache contains some topical QP/MS packs whose displayed
question numbers differ even though their order and marks correspond.

Therefore:

```text
exact number match
    → auto-valid candidate

same main number match
    → needs human review

sequence + marks alignment
    → needs human review

missing question/MS candidate
    → critical parsing issue
```

No review-aligned result is enabled for retrieval in this notebook.


## 12. Run the pilot or complete 35-pair dry run

Default mode parses only the selected three pilot pairs and does not write to PostgreSQL.


In [56]:
parse_results: list[PairParseResult] = []
runtime_failures: list[dict[str, Any]] = []

for index, source in enumerate(selected_pairs, start=1):
    print(
        f"[{index}/{len(selected_pairs)}] "
        f"Parsing {source['pair_key']}..."
    )

    try:
        result = parse_pair(source)
        parse_results.append(result)

        print(
            f"  scored={result.scored_question_count}, "
            f"MS={result.mark_scheme_entry_count}, "
            f"exact_links={result.exact_link_count}, "
            f"fallback={result.fallback_link_count}, "
            f"passed={result.strict_passed}"
        )

    except Exception as error:
        runtime_failures.append(
            {
                "pair_key": source["pair_key"],
                "question_path": str(source["question_path"]),
                "mark_scheme_path": str(source["mark_scheme_path"]),
                "error_type": type(error).__name__,
                "error": str(error),
            }
        )
        print(
            f"  FAILED: {type(error).__name__}: {error}"
        )


parse_summary_df = pd.DataFrame(
    [
        {
            "pair_key": result.pair_key,
            "topic": result.topic_name,
            "subtopic_code": result.subtopic_code,
            "subtopic": result.subtopic_name,
            "scored_questions": result.scored_question_count,
            "context_records": result.context_count,
            "ms_entries": result.mark_scheme_entry_count,
            "exact_links": result.exact_link_count,
            "fallback_links": result.fallback_link_count,
            "unlinked_questions": result.unlinked_question_count,
            "unlinked_ms_entries": result.unlinked_mark_scheme_count,
            "mark_mismatches": result.mark_mismatch_count,
            "issue_count": len(result.issues),
            "strict_passed": result.strict_passed,
            "needs_human_review": result.needs_human_review,
            "processing_seconds": result.processing_seconds,
        }
        for result in parse_results
    ]
)

runtime_failures_df = pd.DataFrame(runtime_failures)

display(parse_summary_df)

if not runtime_failures_df.empty:
    print("Runtime failures:")
    display(runtime_failures_df)


[1/35] Parsing AQA_GCSE_CS_PMT_T1_1_1_PYTHON...


  scored=49, MS=49, exact_links=49, fallback=0, passed=True
[2/35] Parsing AQA_GCSE_CS_PMT_T1_1_2_PYTHON...
  scored=10, MS=10, exact_links=10, fallback=0, passed=True
[3/35] Parsing AQA_GCSE_CS_PMT_T1_1_3_PYTHON...
  scored=9, MS=8, exact_links=8, fallback=0, passed=False
[4/35] Parsing AQA_GCSE_CS_PMT_T1_1_4_PYTHON...
  scored=13, MS=13, exact_links=13, fallback=0, passed=False
[5/35] Parsing AQA_GCSE_CS_PMT_T2_2_01_PYTHON...
  scored=22, MS=22, exact_links=22, fallback=0, passed=True
[6/35] Parsing AQA_GCSE_CS_PMT_T2_2_02_PYTHON...
  scored=112, MS=112, exact_links=112, fallback=0, passed=False
[7/35] Parsing AQA_GCSE_CS_PMT_T2_2_03_PYTHON...
  scored=19, MS=19, exact_links=18, fallback=1, passed=True
[8/35] Parsing AQA_GCSE_CS_PMT_T2_2_04_PYTHON...
  scored=28, MS=28, exact_links=28, fallback=0, passed=True
[9/35] Parsing AQA_GCSE_CS_PMT_T2_2_05_PYTHON...
  scored=19, MS=19, exact_links=19, fallback=0, passed=True
[10/35] Parsing AQA_GCSE_CS_PMT_T2_2_06_PYTHON...
  scored=31, MS=31

,pair_key,topic,subtopic_code,subtopic,scored_questions,context_records,ms_entries,exact_links,fallback_links,unlinked_questions,unlinked_ms_entries,mark_mismatches,issue_count,strict_passed,needs_human_review,processing_seconds
0,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,Fundamentals of Algorithms,1.1,Representing Algorithms,49,9,49,49,0,0,0,0,1,True,False,6.8633
1,AQA_GCSE_CS_PMT_T1_1_2_PYTHON,Fundamentals of Algorithms,1.2,Efficiency of Algorithms,10,3,10,10,0,0,0,0,0,True,False,0.9325
2,AQA_GCSE_CS_PMT_T1_1_3_PYTHON,Fundamentals of Algorithms,1.3,Searching Algorithms,9,0,8,8,0,1,0,0,3,False,True,1.1042
3,AQA_GCSE_CS_PMT_T1_1_4_PYTHON,Fundamentals of Algorithms,1.4,Sorting Algorithms,13,2,13,13,0,0,0,1,1,False,True,1.5162
4,AQA_GCSE_CS_PMT_T2_2_01_PYTHON,Programming,2.01,Data Types,22,7,22,22,0,0,0,0,0,True,False,1.8903
5,AQA_GCSE_CS_PMT_T2_2_02_PYTHON,Programming,2.02,Programming Concepts,112,11,112,112,0,0,0,1,2,False,True,10.7002
6,AQA_GCSE_CS_PMT_T2_2_03_PYTHON,Programming,2.03,Arithmetic Operations,19,4,19,18,1,0,0,0,1,True,True,2.7196
7,AQA_GCSE_CS_PMT_T2_2_04_PYTHON,Programming,2.04,Relational Operations,28,6,28,28,0,0,0,0,1,True,False,1.3451
8,AQA_GCSE_CS_PMT_T2_2_05_PYTHON,Programming,2.05,Boolean Operations,19,4,19,19,0,0,0,0,0,True,False,1.0516
9,AQA_GCSE_CS_PMT_T2_2_06_PYTHON,Programming,2.06,Data Structures,31,7,31,31,0,0,0,0,0,True,False,3.2208


## 13. Inspect parsed questions, mark-scheme entries, links, and issues

Do not enable full batch until these pilot tables look correct.


In [57]:
question_rows: list[dict[str, Any]] = []
ms_rows: list[dict[str, Any]] = []
link_rows: list[dict[str, Any]] = []
issue_rows: list[dict[str, Any]] = []

for result in parse_results:
    for question in result.questions:
        question_rows.append(
            {
                "topic": result.topic_name,
                "subtopic": result.subtopic_name,
                **question.model_dump(mode="json"),
            }
        )

    for entry in result.mark_scheme_entries:
        ms_rows.append(
            {
                "topic": result.topic_name,
                "subtopic": result.subtopic_name,
                **entry.model_dump(mode="json"),
            }
        )

    for link in result.links:
        link_rows.append(
            link.model_dump(mode="json")
        )

    for issue in result.issues:
        issue_rows.append(
            {
                "pair_key": result.pair_key,
                **make_json_safe(issue),
            }
        )

parsed_questions_df = pd.DataFrame(question_rows)
parsed_mark_schemes_df = pd.DataFrame(ms_rows)
proposed_links_df = pd.DataFrame(link_rows)
parsing_issues_df = pd.DataFrame(issue_rows)

print("Parsed questions:")
display(
    parsed_questions_df[
        [
            column
            for column in [
                "pair_key",
                "sequence_index",
                "question_number",
                "occurrence_index",
                "record_type",
                "marks",
                "page_start",
                "page_end",
                "has_visual",
                "has_code",
                "specification_scope",
                "review_status",
                "parse_warnings",
                "question_text",
            ]
            if column in parsed_questions_df.columns
        ]
    ]
    if not parsed_questions_df.empty
    else parsed_questions_df
)

print("Parsed mark-scheme entries:")
display(
    parsed_mark_schemes_df[
        [
            column
            for column in [
                "pair_key",
                "sequence_index",
                "question_number",
                "occurrence_index",
                "maximum_marks",
                "page_start",
                "page_end",
                "specification_scope",
                "review_status",
                "parse_warnings",
                "marking_guidance",
            ]
            if column in parsed_mark_schemes_df.columns
        ]
    ]
    if not parsed_mark_schemes_df.empty
    else parsed_mark_schemes_df
)

print("Proposed QP/MS links:")
display(proposed_links_df)

print("Parsing issues:")
display(parsing_issues_df)


Parsed questions:


,pair_key,sequence_index,question_number,occurrence_index,record_type,marks,page_start,page_end,has_visual,has_code,specification_scope,review_status,parse_warnings,question_text
0,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,1,01.1,1,scored_item,2.0,1,1,False,False,source_not_explicit_in_extracted_text,auto_valid,[],Define the term algorithm.
1,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,2,01.2,1,scored_item,3.0,1,2,False,True,source_not_explicit_in_extracted_text,auto_valid,[],The following are computer science terms (labe...
2,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,3,02,1,scored_item,7.0,2,4,True,True,source_not_explicit_in_extracted_text,auto_valid,[visual_asset_review_recommended],The subroutine CHAR_TO_CODE(character) returns...
3,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,4,03,1,scored_item,8.0,4,5,True,True,source_not_explicit_in_extracted_text,auto_valid,[visual_asset_review_recommended],Develop an algorithm using either pseudo-code ...
4,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,5,04,1,context,NaN,6,6,True,False,source_not_explicit_in_extracted_text,auto_valid,[visual_asset_review_recommended],The following subroutines control the way that...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
937,AQA_GCSE_CS_PMT_T8_8_THEORY,2,02,1,scored_item,4.0,2,3,False,False,source_not_explicit_in_extracted_text,auto_valid,[],Explain two reasons why software companies usu...
938,AQA_GCSE_CS_PMT_T8_8_THEORY,3,03,1,scored_item,4.0,3,4,False,False,source_not_explicit_in_extracted_text,auto_valid,[],A healthcare publication contains the followin...
939,AQA_GCSE_CS_PMT_T8_8_THEORY,4,04,1,scored_item,6.0,5,5,False,True,source_not_explicit_in_extracted_text,auto_valid,[],An autonomous vehicle is controlled by a compu...
940,AQA_GCSE_CS_PMT_T8_8_THEORY,5,05,1,scored_item,9.0,6,6,False,False,source_not_explicit_in_extracted_text,auto_valid,[],"Wearable devices, such as smartwatches and fit..."


Parsed mark-scheme entries:


,pair_key,sequence_index,question_number,occurrence_index,maximum_marks,page_start,page_end,specification_scope,review_status,parse_warnings,marking_guidance
0,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,1,01.1,1,2,1,1,source_not_explicit_in_extracted_text,auto_valid,"[alignment_method=exact_number_occurrence, ali...",2 marks for AO1 (recall)\nA sequence/number/se...
1,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,2,01.2,1,3,1,1,source_not_explicit_in_extracted_text,auto_valid,"[alignment_method=exact_number_occurrence, ali...",3 marks for AO1 (recall)\nOne mark for each co...
2,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,3,02,1,7,2,4,source_not_explicit_in_extracted_text,auto_valid,"[alignment_method=exact_number_occurrence, ali...",7 marks for AO3 (program)\nIf CHAR_TO_CODE is ...
3,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,4,03,1,8,5,7,source_not_explicit_in_extracted_text,auto_valid,"[alignment_method=exact_number_occurrence, ali...",8 marks for AO3 (program)\nDPT. For repeated e...
4,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,5,04.1,1,3,8,8,source_not_explicit_in_extracted_text,auto_valid,"[alignment_method=exact_number_occurrence, ali...",3 marks for AO2 (apply)\n1 mark for C written ...
...,...,...,...,...,...,...,...,...,...,...,...
816,AQA_GCSE_CS_PMT_T8_8_THEORY,2,02,1,4,2,2,source_not_explicit_in_extracted_text,auto_valid,"[alignment_method=exact_number_occurrence, ali...",4 marks for AO2 (apply)\n1 mark for stating ea...
817,AQA_GCSE_CS_PMT_T8_8_THEORY,3,03,1,4,3,3,source_not_explicit_in_extracted_text,auto_valid,"[alignment_method=exact_number_occurrence, ali...",2 marks for AO1 (understanding) and 2 marks fo...
818,AQA_GCSE_CS_PMT_T8_8_THEORY,4,04,1,6,4,4,source_not_explicit_in_extracted_text,auto_valid,"[alignment_method=exact_number_occurrence, ali...",6 marks for AO2 (apply)\nMark\nLevel Descripti...
819,AQA_GCSE_CS_PMT_T8_8_THEORY,5,05,1,9,5,7,source_not_explicit_in_extracted_text,auto_valid,"[alignment_method=exact_number_occurrence, ali...",9 marks for AO2 (apply)\nLevel Description Mar...


Proposed QP/MS links:


,question_internal_key,mark_scheme_internal_key,pair_key,match_method,match_confidence,marks_match,validation_status,validation_warnings
0,1.1#1,1.1#1,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,exact_number_occurrence,1.0,True,auto_valid,[]
1,1.2#1,1.2#1,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,exact_number_occurrence,1.0,True,auto_valid,[]
2,2#1,2#1,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,exact_number_occurrence,1.0,True,auto_valid,[]
3,3#1,3#1,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,exact_number_occurrence,1.0,True,auto_valid,[]
4,4.1#1,4.1#1,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,exact_number_occurrence,1.0,True,auto_valid,[]
...,...,...,...,...,...,...,...,...
816,2#1,2#1,AQA_GCSE_CS_PMT_T8_8_THEORY,exact_number_occurrence,1.0,True,auto_valid,[]
817,3#1,3#1,AQA_GCSE_CS_PMT_T8_8_THEORY,exact_number_occurrence,1.0,True,auto_valid,[]
818,4#1,4#1,AQA_GCSE_CS_PMT_T8_8_THEORY,exact_number_occurrence,1.0,True,auto_valid,[]
819,5#1,5#1,AQA_GCSE_CS_PMT_T8_8_THEORY,exact_number_occurrence,1.0,True,auto_valid,[]


Parsing issues:


,pair_key,stage,severity,issue_type,description,document_id,details
0,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,question_validation,info,non_question_data_rows_discarded,Numeric/date table rows that resembled questio...,f63642ec-794e-465f-8623-5d6e9693be56,"{'segments': [{'question_number': '60', 'page_..."
1,AQA_GCSE_CS_PMT_T1_1_3_PYTHON,question_validation,info,repeated_question_numbers_supported,Repeated question numbers were retained using ...,05b44ab7-0434-4009-b07e-ae51af5578bb,{'counts': {'3.1': 2}}
2,AQA_GCSE_CS_PMT_T1_1_3_PYTHON,mark_scheme_alignment,critical,questions_without_ms_candidate,One or more scored QP questions could not be a...,5c55fcfa-ff27-441b-af7a-46223676500d,{'question_keys': ['3.1#1']}
3,AQA_GCSE_CS_PMT_T1_1_3_PYTHON,question_mark_scheme_linking,critical,unlinked_questions,One or more scored questions have no linked MS...,05b44ab7-0434-4009-b07e-ae51af5578bb,{'question_keys': ['3.1#1']}
4,AQA_GCSE_CS_PMT_T1_1_4_PYTHON,question_mark_scheme_linking,critical,linked_marks_mismatch,Linked QP and MS mark allocations do not match.,efc7d7bf-c865-41d8-aaee-858617ebcbbe,"{'links': [{'question_key': '1.5#1', 'mark_sch..."
5,AQA_GCSE_CS_PMT_T2_2_02_PYTHON,question_validation,info,non_question_data_rows_discarded,Numeric/date table rows that resembled questio...,c2b7185e-4baf-4bbb-8cc4-b4bc32df37e6,"{'segments': [{'question_number': '21', 'page_..."
6,AQA_GCSE_CS_PMT_T2_2_02_PYTHON,question_mark_scheme_linking,critical,linked_marks_mismatch,Linked QP and MS mark allocations do not match.,c2b7185e-4baf-4bbb-8cc4-b4bc32df37e6,"{'links': [{'question_key': '3.5#1', 'mark_sch..."
7,AQA_GCSE_CS_PMT_T2_2_03_PYTHON,question_mark_scheme_linking,warning,number_alignment_review_queue,Some QP/MS links were aligned by main number o...,3605775b-7ad2-4edf-9d7e-5f0126f6a070,"{'links': [{'question_internal_key': '16.2#1',..."
8,AQA_GCSE_CS_PMT_T2_2_04_PYTHON,question_validation,info,non_question_data_rows_discarded,Numeric/date table rows that resembled questio...,46eaedf3-cb7e-43c1-bb8d-df81b96ba76c,"{'segments': [{'question_number': '60', 'page_..."
9,AQA_GCSE_CS_PMT_T2_2_07_PYTHON,question_validation,info,non_question_data_rows_discarded,Numeric/date table rows that resembled questio...,f98c2845-1f79-4dad-a357-c564df21ec9c,"{'segments': [{'question_number': '60', 'page_..."


## 14. Linked question + marking-guidance preview

This is the most important pilot table. Check that each question has the correct answer guidance.


In [58]:
preview_rows: list[dict[str, Any]] = []

for result in parse_results:
    question_by_key = {
        question.internal_key: question
        for question in result.questions
    }
    entry_by_key = {
        entry.internal_key: entry
        for entry in result.mark_scheme_entries
    }

    for link in result.links:
        question = question_by_key[link.question_internal_key]
        entry = entry_by_key[link.mark_scheme_internal_key]

        preview_rows.append(
            {
                "pair_key": result.pair_key,
                "subtopic": result.subtopic_name,
                "question_key": question.internal_key,
                "question_number": question.question_number,
                "question_marks": question.marks,
                "ms_marks": entry.maximum_marks,
                "marks_match": link.marks_match,
                "match_method": link.match_method,
                "validation_status": link.validation_status,
                "is_legacy": question.is_legacy,
                "question_text": question.question_text,
                "marking_guidance": entry.marking_guidance,
            }
        )

linked_preview_df = pd.DataFrame(preview_rows)
display(linked_preview_df)


,pair_key,subtopic,question_key,question_number,question_marks,ms_marks,marks_match,match_method,validation_status,is_legacy,question_text,marking_guidance
0,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,Representing Algorithms,1.1#1,01.1,2,2,True,exact_number_occurrence,auto_valid,False,Define the term algorithm.,2 marks for AO1 (recall)\nA sequence/number/se...
1,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,Representing Algorithms,1.2#1,01.2,3,3,True,exact_number_occurrence,auto_valid,False,The following are computer science terms (labe...,3 marks for AO1 (recall)\nOne mark for each co...
2,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,Representing Algorithms,2#1,02,7,7,True,exact_number_occurrence,auto_valid,False,The subroutine CHAR_TO_CODE(character) returns...,7 marks for AO3 (program)\nIf CHAR_TO_CODE is ...
3,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,Representing Algorithms,3#1,03,8,8,True,exact_number_occurrence,auto_valid,False,Develop an algorithm using either pseudo-code ...,8 marks for AO3 (program)\nDPT. For repeated e...
4,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,Representing Algorithms,4.1#1,04.1,3,3,True,exact_number_occurrence,auto_valid,False,"This is how the blocks A, B and C are arranged...",3 marks for AO2 (apply)\n1 mark for C written ...
...,...,...,...,...,...,...,...,...,...,...,...,...
816,AQA_GCSE_CS_PMT_T8_8_THEORY,"Ethical, Legal and Environmental Impacts of Di...",2#1,02,4,4,True,exact_number_occurrence,auto_valid,False,Explain two reasons why software companies usu...,4 marks for AO2 (apply)\n1 mark for stating ea...
817,AQA_GCSE_CS_PMT_T8_8_THEORY,"Ethical, Legal and Environmental Impacts of Di...",3#1,03,4,4,True,exact_number_occurrence,auto_valid,False,A healthcare publication contains the followin...,2 marks for AO1 (understanding) and 2 marks fo...
818,AQA_GCSE_CS_PMT_T8_8_THEORY,"Ethical, Legal and Environmental Impacts of Di...",4#1,04,6,6,True,exact_number_occurrence,auto_valid,False,An autonomous vehicle is controlled by a compu...,6 marks for AO2 (apply)\nMark\nLevel Descripti...
819,AQA_GCSE_CS_PMT_T8_8_THEORY,"Ethical, Legal and Environmental Impacts of Di...",5#1,05,9,9,True,exact_number_occurrence,auto_valid,False,"Wearable devices, such as smartwatches and fit...",9 marks for AO2 (apply)\nLevel Description Mar...


## 15. Pilot acceptance checks

A strict pilot pass requires:

- no runtime failures;
- every selected pair parsed;
- no unlinked scored question;
- no unlinked mark-scheme entry;
- no mark mismatch;
- no sequence fallback.

Legacy `8520` records may still correctly require review.


In [59]:
pilot_checks = {
    "all_selected_pairs_returned": (
        len(parse_results) == len(selected_pairs)
    ),
    "no_runtime_failures": runtime_failures_df.empty,
    "all_pairs_have_questions": (
        not parse_summary_df.empty
        and parse_summary_df["scored_questions"].gt(0).all()
    ),
    "all_pairs_have_ms_entries": (
        not parse_summary_df.empty
        and parse_summary_df["ms_entries"].gt(0).all()
    ),
    "no_unlinked_questions": (
        not parse_summary_df.empty
        and parse_summary_df["unlinked_questions"].eq(0).all()
    ),
    "no_unlinked_ms_entries": (
        not parse_summary_df.empty
        and parse_summary_df["unlinked_ms_entries"].eq(0).all()
    ),
    "no_mark_mismatches": (
        not parse_summary_df.empty
        and parse_summary_df["mark_mismatches"].eq(0).all()
    ),
    "all_links_accounted_for": (
        not parse_summary_df.empty
        and (
            parse_summary_df["exact_links"]
            + parse_summary_df["fallback_links"]
        ).eq(
            parse_summary_df["ms_entries"]
        ).all()
    ),
}

pilot_checks = {
    key: bool(value)
    for key, value in pilot_checks.items()
}

pilot_checks_df = pd.DataFrame(
    [
        {"check": key, "passed": value}
        for key, value in pilot_checks.items()
    ]
)

display(pilot_checks_df)

pilot_ready_for_batch = all(pilot_checks.values())

print(f"Pilot ready for full batch: {pilot_ready_for_batch}")

if pilot_ready_for_batch and not RUN_FULL_BATCH:
    print("\nNext:")
    print("1. Manually inspect linked_preview_df.")
    print("2. Set RUN_FULL_BATCH = True.")
    print("3. Keep COMMIT_TO_POSTGRES = False; ""review-aligned records will move to Notebook 03.")
    print("4. Run all cells for a complete dry run.")


,check,passed
0,all_selected_pairs_returned,True
1,no_runtime_failures,True
2,all_pairs_have_questions,True
3,all_pairs_have_ms_entries,True
4,no_unlinked_questions,False
5,no_unlinked_ms_entries,True
6,no_mark_mismatches,False
7,all_links_accounted_for,True


Pilot ready for full batch: False


## 16. Export dry-run reports


In [60]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_label = "batch" if RUN_FULL_BATCH else "pilot"

summary_path = (
    OUTPUT_DIR
    / f"pmt_topical_parsing_{run_label}_summary_{timestamp}.csv"
)
questions_path = (
    OUTPUT_DIR
    / f"pmt_topical_parsed_questions_{run_label}_{timestamp}.csv"
)
mark_schemes_path = (
    OUTPUT_DIR
    / f"pmt_topical_parsed_mark_schemes_{run_label}_{timestamp}.csv"
)
links_path = (
    OUTPUT_DIR
    / f"pmt_topical_proposed_links_{run_label}_{timestamp}.csv"
)
issues_path = (
    OUTPUT_DIR
    / f"pmt_topical_parsing_issues_{run_label}_{timestamp}.csv"
)
runtime_path = (
    OUTPUT_DIR
    / f"pmt_topical_runtime_failures_{run_label}_{timestamp}.csv"
)
json_path = (
    OUTPUT_DIR
    / f"pmt_topical_parsing_{run_label}_summary_{timestamp}.json"
)

parse_summary_df.to_csv(summary_path, index=False)
parsed_questions_df.to_csv(questions_path, index=False)
parsed_mark_schemes_df.to_csv(mark_schemes_path, index=False)
proposed_links_df.to_csv(links_path, index=False)
parsing_issues_df.to_csv(issues_path, index=False)
runtime_failures_df.to_csv(runtime_path, index=False)

json_payload = make_json_safe(
    {
        "generated_at_utc": utc_now().isoformat(),
        "parser_version": PARSER_VERSION,
        "run_mode": run_label,
        "committed": COMMIT_TO_POSTGRES,
        "selected_pair_count": len(selected_pairs),
        "parsed_pair_count": len(parse_results),
        "pilot_checks": pilot_checks,
        "pilot_ready_for_batch": pilot_ready_for_batch,
        "summary_records": parse_summary_df.to_dict(orient="records"),
        "runtime_failures": runtime_failures,
        "output_files": {
            "summary": str(summary_path.resolve()),
            "questions": str(questions_path.resolve()),
            "mark_schemes": str(mark_schemes_path.resolve()),
            "links": str(links_path.resolve()),
            "issues": str(issues_path.resolve()),
            "runtime_failures": str(runtime_path.resolve()),
        },
    }
)

json_path.write_text(
    json.dumps(
        json_payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("Saved:")
print(summary_path)
print(questions_path)
print(mark_schemes_path)
print(links_path)
print(issues_path)
print(runtime_path)
print(json_path)


Saved:
C:\Users\hp\EDTECH\Agent2\OUTPUT\pmt_topical_parsing_batch_summary_20260804_194253.csv
C:\Users\hp\EDTECH\Agent2\OUTPUT\pmt_topical_parsed_questions_batch_20260804_194253.csv
C:\Users\hp\EDTECH\Agent2\OUTPUT\pmt_topical_parsed_mark_schemes_batch_20260804_194253.csv
C:\Users\hp\EDTECH\Agent2\OUTPUT\pmt_topical_proposed_links_batch_20260804_194253.csv
C:\Users\hp\EDTECH\Agent2\OUTPUT\pmt_topical_parsing_issues_batch_20260804_194253.csv
C:\Users\hp\EDTECH\Agent2\OUTPUT\pmt_topical_runtime_failures_batch_20260804_194253.csv
C:\Users\hp\EDTECH\Agent2\OUTPUT\pmt_topical_parsing_batch_summary_20260804_194253.json


## 17. PostgreSQL replacement helpers

The commit is idempotent. Human-approved or human-corrected rows are protected from automatic replacement.


In [61]:
def pair_has_human_reviewed_rows(
    session: Session,
    pair_key: str,
) -> bool:
    reviewed_question_count = (
        session.scalar(
            select(func.count())
            .select_from(AssessmentTopicalQuestion)
            .where(
                AssessmentTopicalQuestion.pair_key == pair_key,
                AssessmentTopicalQuestion.review_status.in_(
                    ["human_approved", "human_corrected"]
                ),
            )
        )
        or 0
    )

    reviewed_ms_count = (
        session.scalar(
            select(func.count())
            .select_from(AssessmentTopicalMarkSchemeEntry)
            .where(
                AssessmentTopicalMarkSchemeEntry.pair_key == pair_key,
                AssessmentTopicalMarkSchemeEntry.review_status.in_(
                    ["human_approved", "human_corrected"]
                ),
            )
        )
        or 0
    )

    return bool(reviewed_question_count or reviewed_ms_count)


def replace_pair_records(
    session: Session,
    result: PairParseResult,
) -> dict[str, int]:
    if (
        pair_has_human_reviewed_rows(session, result.pair_key)
        and not ALLOW_REPLACE_HUMAN_REVIEWED
    ):
        raise RuntimeError(
            f"Human-reviewed records exist for {result.pair_key}; "
            "automatic replacement was blocked."
        )

    existing_count = (
        session.scalar(
            select(func.count())
            .select_from(AssessmentTopicalQuestion)
            .where(AssessmentTopicalQuestion.pair_key == result.pair_key)
        )
        or 0
    )

    if existing_count and not ALLOW_REPLACE_EXISTING:
        raise RuntimeError(
            f"Existing parser records found for {result.pair_key}, "
            "but ALLOW_REPLACE_EXISTING=False."
        )

    session.execute(
        delete(AssessmentTopicalQuestionMarkSchemeLink).where(
            AssessmentTopicalQuestionMarkSchemeLink.pair_key
            == result.pair_key
        )
    )
    session.execute(
        delete(AssessmentTopicalParsingIssue).where(
            AssessmentTopicalParsingIssue.pair_key == result.pair_key
        )
    )
    session.execute(
        delete(AssessmentTopicalQuestion).where(
            AssessmentTopicalQuestion.pair_key == result.pair_key
        )
    )
    session.execute(
        delete(AssessmentTopicalMarkSchemeEntry).where(
            AssessmentTopicalMarkSchemeEntry.pair_key == result.pair_key
        )
    )

    question_ids: dict[str, uuid.UUID] = {}

    for parsed in result.questions:
        record = AssessmentTopicalQuestion(
            question_uid=parsed.question_uid,
            pair_key=parsed.pair_key,
            topic_id=parsed.topic_id,
            question_document_id=parsed.question_document_id,
            sequence_index=parsed.sequence_index,
            question_number=parsed.question_number,
            normalized_question_number=parsed.normalized_question_number,
            occurrence_index=parsed.occurrence_index,
            main_question_number=parsed.main_question_number,
            part_number=parsed.part_number,
            parent_question_number=parsed.parent_question_number,
            record_type=parsed.record_type,
            question_text=parsed.question_text,
            context_text=parsed.context_text,
            search_text=parsed.search_text,
            raw_extracted_text=parsed.raw_extracted_text,
            marks=parsed.marks,
            page_start=parsed.page_start,
            page_end=parsed.page_end,
            has_visual=parsed.has_visual,
            visual_page_numbers=parsed.visual_page_numbers,
            has_code=parsed.has_code,
            specification_scope=parsed.specification_scope,
            is_legacy=parsed.is_legacy,
            parse_warnings=parsed.parse_warnings,
            review_status=parsed.review_status,
            retrieval_enabled=False,
            embedding_status="blocked_pending_review",
            content_hash=parsed.content_hash,
            parse_version=parsed.parse_version,
            is_active=True,
        )

        session.add(record)
        session.flush()
        question_ids[parsed.internal_key] = record.id

    ms_ids: dict[str, uuid.UUID] = {}

    for parsed in result.mark_scheme_entries:
        record = AssessmentTopicalMarkSchemeEntry(
            mark_scheme_uid=parsed.mark_scheme_uid,
            pair_key=parsed.pair_key,
            topic_id=parsed.topic_id,
            mark_scheme_document_id=parsed.mark_scheme_document_id,
            sequence_index=parsed.sequence_index,
            question_number=parsed.question_number,
            normalized_question_number=parsed.normalized_question_number,
            occurrence_index=parsed.occurrence_index,
            main_question_number=parsed.main_question_number,
            part_number=parsed.part_number,
            maximum_marks=parsed.maximum_marks,
            marking_guidance=parsed.marking_guidance,
            marking_points=parsed.marking_points,
            acceptable_answers=parsed.acceptable_answers,
            rejected_answers=parsed.rejected_answers,
            additional_guidance=parsed.additional_guidance,
            assessment_objectives=parsed.assessment_objectives,
            page_start=parsed.page_start,
            page_end=parsed.page_end,
            raw_extracted_text=parsed.raw_extracted_text,
            specification_scope=parsed.specification_scope,
            is_legacy=parsed.is_legacy,
            parse_warnings=parsed.parse_warnings,
            review_status=parsed.review_status,
            content_hash=parsed.content_hash,
            parse_version=parsed.parse_version,
            is_active=True,
        )

        session.add(record)
        session.flush()
        ms_ids[parsed.internal_key] = record.id

    for proposed in result.links:
        question_id = question_ids.get(proposed.question_internal_key)
        ms_id = ms_ids.get(proposed.mark_scheme_internal_key)

        if question_id is None or ms_id is None:
            raise RuntimeError(
                "A proposed link references a missing inserted record."
            )

        session.add(
            AssessmentTopicalQuestionMarkSchemeLink(
                pair_key=proposed.pair_key,
                question_id=question_id,
                mark_scheme_entry_id=ms_id,
                match_method=proposed.match_method,
                match_confidence=proposed.match_confidence,
                marks_match=proposed.marks_match,
                validation_status=proposed.validation_status,
                validation_warnings=proposed.validation_warnings,
            )
        )

    for issue in result.issues:
        session.add(
            AssessmentTopicalParsingIssue(
                pair_key=result.pair_key,
                document_id=issue.get("document_id"),
                stage=issue["stage"],
                issue_type=issue["issue_type"],
                severity=issue["severity"],
                description=issue["description"],
                details=make_json_safe(issue.get("details", {})),
            )
        )

    document_status = (
        "parsed"
        if result.strict_passed
        else "needs_review"
    )

    for document_id in (
        result.question_document_id,
        result.mark_scheme_document_id,
    ):
        document = session.get(
            AssessmentTopicalDocument,
            document_id,
        )
        if document is not None:
            document.parsing_status = document_status
            document.error_message = None
            document.updated_at = utc_now()

    return {
        "questions": len(result.questions),
        "mark_scheme_entries": len(result.mark_scheme_entries),
        "links": len(result.links),
        "issues": len(result.issues),
    }


## 18. Optional PostgreSQL commit

Recommended sequence:

1. Pilot dry run.
2. Full 35-pair dry run.
3. Review summaries.
4. Set `COMMIT_TO_POSTGRES = True`.
5. Run again.


In [62]:
commit_summary_rows: list[dict[str, Any]] = []

if not COMMIT_TO_POSTGRES:
    print(
        "Database commit skipped because "
        "COMMIT_TO_POSTGRES=False."
    )

else:
    run_mode = "batch" if RUN_FULL_BATCH else "pilot"

    with Session(engine) as session:
        parsing_run = AssessmentTopicalParsingRun(
            parser_version=PARSER_VERSION,
            run_mode=run_mode,
            committed=True,
            status="running",
            counts={},
        )
        session.add(parsing_run)
        session.commit()
        parsing_run_id = parsing_run.id

    aggregate_counts = {
        "pairs_attempted": len(parse_results),
        "pairs_committed": 0,
        "pairs_failed": 0,
        "questions": 0,
        "mark_scheme_entries": 0,
        "links": 0,
        "issues": 0,
    }

    for result in parse_results:
        try:
            with Session(engine) as session:
                counts = replace_pair_records(
                    session,
                    result,
                )
                session.commit()

            aggregate_counts["pairs_committed"] += 1

            for key in (
                "questions",
                "mark_scheme_entries",
                "links",
                "issues",
            ):
                aggregate_counts[key] += counts[key]

            commit_summary_rows.append(
                {
                    "pair_key": result.pair_key,
                    "status": "committed",
                    **counts,
                }
            )

        except Exception as error:
            aggregate_counts["pairs_failed"] += 1
            commit_summary_rows.append(
                {
                    "pair_key": result.pair_key,
                    "status": "failed",
                    "error": str(error),
                }
            )

    with Session(engine) as session:
        parsing_run = session.get(
            AssessmentTopicalParsingRun,
            parsing_run_id,
        )

        if parsing_run is not None:
            parsing_run.status = (
                "completed"
                if aggregate_counts["pairs_failed"] == 0
                else "completed_with_failures"
            )
            parsing_run.completed_at = utc_now()
            parsing_run.counts = make_json_safe(
                aggregate_counts
            )
            session.commit()

commit_summary_df = pd.DataFrame(
    commit_summary_rows
)
display(commit_summary_df)


,pair_key,status,questions,mark_scheme_entries,links,issues
0,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,committed,58,49,49,1
1,AQA_GCSE_CS_PMT_T1_1_2_PYTHON,committed,13,10,10,0
2,AQA_GCSE_CS_PMT_T1_1_3_PYTHON,committed,9,8,8,3
3,AQA_GCSE_CS_PMT_T1_1_4_PYTHON,committed,15,13,13,1
4,AQA_GCSE_CS_PMT_T2_2_01_PYTHON,committed,29,22,22,0
5,AQA_GCSE_CS_PMT_T2_2_02_PYTHON,committed,123,112,112,2
6,AQA_GCSE_CS_PMT_T2_2_03_PYTHON,committed,23,19,19,1
7,AQA_GCSE_CS_PMT_T2_2_04_PYTHON,committed,34,28,28,1
8,AQA_GCSE_CS_PMT_T2_2_05_PYTHON,committed,23,19,19,0
9,AQA_GCSE_CS_PMT_T2_2_06_PYTHON,committed,38,31,31,0


## 19. Final PostgreSQL smoke checks


In [63]:
with Session(engine) as session:
    database_counts = {
        "topical_topics": (
            session.scalar(
                select(func.count())
                .select_from(AssessmentTopicalTopic)
            )
            or 0
        ),
        "topical_documents": (
            session.scalar(
                select(func.count())
                .select_from(AssessmentTopicalDocument)
            )
            or 0
        ),
        "topical_questions": (
            session.scalar(
                select(func.count())
                .select_from(AssessmentTopicalQuestion)
            )
            or 0
        ),
        "topical_ms_entries": (
            session.scalar(
                select(func.count())
                .select_from(AssessmentTopicalMarkSchemeEntry)
            )
            or 0
        ),
        "topical_links": (
            session.scalar(
                select(func.count())
                .select_from(AssessmentTopicalQuestionMarkSchemeLink)
            )
            or 0
        ),
        "topical_open_issues": (
            session.scalar(
                select(func.count())
                .select_from(AssessmentTopicalParsingIssue)
                .where(
                    AssessmentTopicalParsingIssue.resolved.is_(False)
                )
            )
            or 0
        ),
        "retrieval_enabled_questions": (
            session.scalar(
                select(func.count())
                .select_from(AssessmentTopicalQuestion)
                .where(
                    AssessmentTopicalQuestion.retrieval_enabled.is_(True)
                )
            )
            or 0
        ),
    }

database_counts_df = pd.DataFrame(
    [
        {"metric": key, "value": value}
        for key, value in database_counts.items()
    ]
)

display(database_counts_df)

print(
    "retrieval_enabled_questions must remain 0 "
    "until Notebook 03 human review."
)


,metric,value
0,topical_topics,35
1,topical_documents,70
2,topical_questions,942
3,topical_ms_entries,821
4,topical_links,821
5,topical_open_issues,20
6,retrieval_enabled_questions,0


retrieval_enabled_questions must remain 0 until Notebook 03 human review.


# Notebook 02 completion path

### First run

```text
RUN_FULL_BATCH = False
COMMIT_TO_POSTGRES = False
```

Inspect:

```text
parse_summary_df
parsed_questions_df
parsed_mark_schemes_df
linked_preview_df
pilot_checks_df
```

### Second run

```text
RUN_FULL_BATCH = True
COMMIT_TO_POSTGRES = False
```

Review all 35-pair dry-run summaries.

### Final run

```text
RUN_FULL_BATCH = True
COMMIT_TO_POSTGRES = True
```

Then confirm:

```text
topical_questions > 0
topical_ms_entries > 0
topical_links > 0
retrieval_enabled_questions = 0
```

The next notebook is:

```text
03_parsing_quality_and_human_review.ipynb
```
